# Taxila CHIP: executable Q1 analysis, uncertainty audit and publication figures

**Climate-linked Heritage Inspection Prioritisation (CHIP) for the UNESCO World Heritage Site of Taxila, Pakistan**

This self-contained notebook reconstructs the frozen evidence chain, recomputes the primary 500 m prioritisation, runs scale/weight/spatial/structural uncertainty experiments, benchmarks the WorldCover-derived proxy classifier, creates journal-quality raster and vector figures, and checks headline claims against a machine-readable evidence lock.

> **Inference boundary.** CHIP ranks **relative field-inspection priority**. It does **not** confirm damage, deterioration, causal hazard, legal status, cadastral limits, or UNESCO property/buffer boundaries. The circles used below (250/500/1000 m) are analytic supports only.

## tl;dr

- The frozen inventory contains **18 official component records; 17 have usable mapped points**. Saraikala (139-002) remains unresolved and is not silently geocoded.
- Five post-monsoon Landsat epochs are analysed (2004, 2009, 2014, 2019, 2024; **55 retained scenes**). E2004–E2024 common spectral support is expected to be **386,280 / 403,480 cells (95.737%)**.
- The primary score is $P_i=0.5L_i+0.5T_i$ at 500 m. Ranking uncertainty is reported through shared Bayesian blocks, coordinate jitter with terrain re-extraction, domain-weight concentration, structural scenarios, and joint spatial × decision draws.
- Climate is contextual: 1991–2025 annual precipitation has a negative Theil–Sen trend and mean temperature a positive trend in the frozen moving-block analysis. These do not establish site-condition causality.
- WorldCover labels provide a **spatially buffered proxy agreement test**, not heritage-condition validation.

Run all cells top-to-bottom. The default `publication` profile uses the manuscript settings and can be compute intensive. `validation` is a shorter engineering check.

**Required companion file:** `Taxila_CHIP_Colab_Runtime_Data.zip`. In Colab, Cell 6 opens an upload chooser if no valid archive is present. The loader accepts Chrome-style `(1).zip` renames, verifies the ZIP container and CRCs, and refuses incomplete/HTML downloads with a diagnostic message.

## 1. Runtime, profile and reproducibility controls

The analysis is deterministic where possible (`SEED=311`). Stochastic summaries use fixed seeds and report draw counts. Most geospatial and scikit-learn operations are CPU-bound; selecting a Colab/Kaggle GPU runtime is compatible but does not materially accelerate the unchanged manuscript pipeline.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "rasterio": "rasterio",
    "PIL": "Pillow",
    "joblib": "joblib",
}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

PROFILE = os.environ.get("TAXILA_PROFILE", "publication").strip().lower()
if PROFILE not in {"publication", "validation"}:
    raise ValueError("TAXILA_PROFILE must be 'publication' or 'validation'.")

PROFILES = {
    "publication": {
        "block_draws": 2_000,
        "jitter_draws": 2_000,
        "weight_draws": 50_000,
        "joint_spatial_states": 500,
        "decision_draws_per_state": 10,
        "climate_draws": 2_000,
        "proxy_bootstrap_draws": 2_000,
        "refit_proxy_models": True,
    },
    "validation": {
        "block_draws": 120,
        "jitter_draws": 40,
        "weight_draws": 5_000,
        "joint_spatial_states": 80,
        "decision_draws_per_state": 5,
        "climate_draws": 250,
        "proxy_bootstrap_draws": 100,
        "refit_proxy_models": False,
    },
}
CONFIG = PROFILES[PROFILE]
print(f"Profile: {PROFILE}")
print(CONFIG)

In [ ]:
from __future__ import annotations

import gzip
import hashlib
import json
import math
import os
import random
import shutil
import sys
import warnings
import zipfile
from collections import defaultdict, deque
from pathlib import Path
from typing import Iterable

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from PIL import Image
from scipy import ndimage
from scipy.special import expit
from scipy.stats import kendalltau, rankdata, spearmanr, theilslopes
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize


SEED = 311
EPOCHS = ["E2004", "E2009", "E2014", "E2019", "E2024"]
EPOCH_YEARS = dict(zip(EPOCHS, [2004, 2009, 2014, 2019, 2024]))
RADII = [250, 500, 1000]
GRID_WIDTH = 616
GRID_HEIGHT = 655
GRID_XMIN = 292_980.0
GRID_YMAX = 3_748_620.0
GRID_CELL = 30.0
GRID_XMAX = GRID_XMIN + GRID_WIDTH * GRID_CELL
GRID_YMIN = GRID_YMAX - GRID_HEIGHT * GRID_CELL
NODATA = -9999.0
INDEX_NAMES = ["NDVI", "NDBI", "MNDWI", "BSI"]
PROXY_CLASS_NAMES = {
    1: "Built-up / impervious",
    2: "Bare / sparse vegetation",
    3: "Cultivated land",
    4: "Woody / dense vegetation",
    5: "Shrub / grass / herbaceous",
    6: "Water",
}

PALETTE = {
    "ink": "#17324D",
    "blue": "#2F6F9F",
    "blue_light": "#A9C9E2",
    "teal": "#2A9D8F",
    "gold": "#D7A23B",
    "orange": "#D97732",
    "pink": "#B35C7A",
    "olive": "#788B3D",
    "grey": "#6B7280",
    "grid": "#DCE3EA",
    "paper": "#FBFCFE",
    "muted": "#EEF2F6",
}


def configure_publication_style() -> None:
    """Apply a restrained, journal-safe visual style."""
    sns.set_theme(style="whitegrid", context="paper")
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9.0,
            "axes.titlesize": 10.5,
            "axes.labelsize": 9.0,
            "axes.titleweight": "semibold",
            "axes.edgecolor": PALETTE["ink"],
            "axes.linewidth": 0.8,
            "xtick.labelsize": 8.0,
            "ytick.labelsize": 8.0,
            "legend.fontsize": 7.8,
            "figure.dpi": 130,
            "savefig.dpi": 450,
            "savefig.bbox": "tight",
            "savefig.facecolor": "white",
            "axes.facecolor": "white",
            "figure.facecolor": "white",
            "grid.color": PALETTE["grid"],
            "grid.linewidth": 0.6,
            "grid.alpha": 0.7,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
        }
    )


def _data_root_missing(path: str | Path) -> list[str]:
    """Return missing runtime inputs; an empty list means the package is executable."""
    root = Path(path)
    required = [
        "02_raw_data/N33E072.hgt.gz",
        "06_processed_vectors/taxila_components_epsg32643.geojson",
        "06_processed_vectors/taxila_components_wgs84.geojson",
        "13_tables/baseline_reproduction/component_epoch_multiscale_landsat.csv",
        "13_tables/baseline_reproduction/component_multiscale_terrain_hydrology.csv",
        "13_tables/baseline_reproduction/component_epoch_integrated_scores.csv",
        "13_tables/baseline_reproduction/proxy_model_feature_table.csv",
        "13_tables/open_meteo_annual_climate_metrics.csv",
        "13_tables/enhanced_spatially_buffered_proxy_model_comparison.csv",
        "14_statistics/open_meteo_climate_trends_block_bootstrap.csv",
        "q1_revision/adverse_tail_threshold_sensitivity.csv",
        "cartography/natural_earth_10m_admin0_pakistan_pov_v5_1_1_region.geojson",
    ]
    required.extend(f"05_processed_rasters/{epoch}_oli_like_indices.tif" for epoch in EPOCHS)
    missing = [relative for relative in required if not (root / relative).is_file()]
    for epoch in EPOCHS:
        alternatives = [
            root / "05_processed_rasters" / f"{epoch}_relative_landscape_pressure.f32",
            root / "05_processed_rasters" / f"{epoch}_relative_landscape_pressure.tif",
        ]
        if not any(path.is_file() for path in alternatives):
            missing.append(f"05_processed_rasters/{epoch}_relative_landscape_pressure.[f32|tif]")
    convergence_alternatives = [
        root / "05_processed_rasters" / "E2004_E2024_spectral_convergence.f32",
        root / "05_processed_rasters" / "E2004_E2024_spectral_convergence.tif",
    ]
    if not any(path.is_file() for path in convergence_alternatives):
        missing.append("05_processed_rasters/E2004_E2024_spectral_convergence.[f32|tif]")
    return missing


def _unique_existing(paths: Iterable[Path]) -> list[Path]:
    output: list[Path] = []
    seen: set[str] = set()
    for path in paths:
        try:
            key = str(path.expanduser().resolve())
        except OSError:
            key = str(path)
        if key not in seen and path.exists():
            seen.add(key)
            output.append(path)
    return output


def _archive_diagnostic(path: Path) -> str:
    size = path.stat().st_size
    with path.open("rb") as stream:
        header = stream.read(16)
    header_text = header.decode("utf-8", errors="replace").replace("\n", " ").replace("\r", " ")
    if header.lstrip().startswith((b"<", b"{")):
        reason = "downloaded web/error content rather than ZIP bytes"
    elif size < 1_000_000:
        reason = "file is far too small and is probably incomplete"
    else:
        reason = "central directory is missing; the upload/download is truncated or corrupted"
    return f"{path} ({size:,} bytes; header={header_text!r}): {reason}"


def _safe_extract_zip(bundle: zipfile.ZipFile, destination: Path) -> None:
    destination_resolved = destination.resolve()
    for member in bundle.infolist():
        member_target = (destination / member.filename).resolve()
        if destination_resolved != member_target and destination_resolved not in member_target.parents:
            raise RuntimeError(f"Unsafe path in data archive: {member.filename}")
    bundle.extractall(destination)


def locate_data_root(explicit: str | Path | None = None) -> Path:
    """Locate, validate and safely extract the curated Colab/Kaggle evidence package."""
    explicit_path = Path(explicit).expanduser() if explicit else None
    env_value = os.environ.get("TAXILA_CHIP_DATA_ROOT")
    env_path = Path(env_value).expanduser() if env_value else None

    folder_candidates: list[Path] = []
    for path in (explicit_path, env_path):
        if path is not None and path.is_dir():
            folder_candidates.append(path)
    search_roots = [Path.cwd(), *Path.cwd().parents]
    for search_root in search_roots:
        folder_candidates.extend(
            [
                search_root / "Taxila_CHIP_Frozen_Evidence_Data",
                search_root / "data" / "Taxila_CHIP_Frozen_Evidence_Data",
            ]
        )
    folder_candidates.extend(
        [
            Path("/content/Taxila_CHIP_Frozen_Evidence_Data"),
            Path("/kaggle/working/Taxila_CHIP_Frozen_Evidence_Data"),
        ]
    )
    if Path("/kaggle/input").exists():
        folder_candidates.extend(Path("/kaggle/input").glob("**/Taxila_CHIP_Frozen_Evidence_Data"))

    invalid_folders: list[str] = []
    for candidate in _unique_existing(folder_candidates):
        missing = _data_root_missing(candidate)
        if not missing:
            return candidate.resolve()
        invalid_folders.append(f"{candidate}: missing {len(missing)} required inputs")

    archive_candidates: list[Path] = []
    for path in (explicit_path, env_path):
        if path is not None and path.is_file():
            archive_candidates.append(path)
    search_roots = [Path.cwd(), Path("/content"), Path("/kaggle/working")]
    patterns = ["Taxila_CHIP_Colab_Runtime_Data*.zip", "Taxila_CHIP_Frozen_Evidence_Data*.zip"]
    for search_root in search_roots:
        if search_root.exists():
            for pattern in patterns:
                archive_candidates.extend(sorted(search_root.glob(pattern)))
    if Path("/kaggle/input").exists():
        for pattern in patterns:
            archive_candidates.extend(sorted(Path("/kaggle/input").glob(f"**/{pattern}")))

    invalid_archives: list[str] = []
    for archive in _unique_existing(archive_candidates):
        if not archive.is_file():
            continue
        if not zipfile.is_zipfile(archive):
            invalid_archives.append(_archive_diagnostic(archive))
            continue
        try:
            archive_hash = sha256(archive)[:12]
            extraction_cache = Path.cwd() / f"_taxila_chip_extracted_{archive_hash}"
            with zipfile.ZipFile(archive) as bundle:
                corrupt_member = bundle.testzip()
                if corrupt_member is not None:
                    invalid_archives.append(f"{archive}: CRC failure in {corrupt_member}")
                    continue
                _safe_extract_zip(bundle, extraction_cache)
            extracted_candidates = [extraction_cache / "Taxila_CHIP_Frozen_Evidence_Data", extraction_cache]
            extracted_candidates.extend(
                path for path in extraction_cache.rglob("Taxila_CHIP_Frozen_Evidence_Data") if path.is_dir()
            )
            for extracted in _unique_existing(extracted_candidates):
                missing = _data_root_missing(extracted)
                if not missing:
                    print(f"Validated data archive: {archive.name} ({archive.stat().st_size:,} bytes)")
                    return extracted.resolve()
            invalid_archives.append(f"{archive}: valid ZIP container but required CHIP runtime files are missing")
        except (OSError, zipfile.BadZipFile, RuntimeError) as error:
            invalid_archives.append(f"{archive}: {type(error).__name__}: {error}")

    details = "\n".join(f"  - {item}" for item in invalid_archives + invalid_folders)
    if details:
        details = "\nFiles/folders inspected:\n" + details
    raise FileNotFoundError(
        "No valid Taxila CHIP data package was found. Re-download and upload "
        "Taxila_CHIP_Colab_Runtime_Data.zip beside the notebook; do not rename an HTML download page to .zip. "
        "A valid runtime ZIP is tens of megabytes and begins with the bytes 'PK'. "
        "Colab may rename a second upload to '(1).zip'; this loader detects that automatically."
        + details
    )


def prepare_output_tree(output_root: str | Path) -> dict[str, Path]:
    root = Path(output_root)
    paths = {
        "root": root,
        "figures": root / "figures",
        "tables": root / "tables",
        "models": root / "models",
        "validation": root / "validation",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_figure(fig: plt.Figure, output_dir: str | Path, stem: str, dpi: int = 450) -> list[Path]:
    """Save vector and high-resolution raster copies and verify both."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs: list[Path] = []
    for extension in ("png", "pdf", "svg"):
        path = output_dir / f"{stem}.{extension}"
        temporary = output_dir / f".{stem}.{extension}.building"
        fig.savefig(
            temporary,
            format=extension,
            dpi=dpi if extension == "png" else None,
            facecolor="white",
        )
        # Windows may reject fsync on a read-only descriptor. Reopen in a
        # writable binary mode so the durability check is cross-platform.
        with temporary.open("r+b") as stream:
            stream.flush()
            os.fsync(stream.fileno())
        if extension == "png":
            with Image.open(temporary) as image:
                image.verify()
        elif extension == "pdf":
            payload = temporary.read_bytes()
            if not payload.startswith(b"%PDF") or b"%%EOF" not in payload[-4096:]:
                raise RuntimeError(f"Invalid PDF export: {temporary}")
        elif "<svg" not in temporary.read_text(encoding="utf-8", errors="ignore")[:2000]:
            raise RuntimeError(f"Invalid SVG export: {temporary}")
        os.replace(temporary, path)
        # POSIX permits fsync on a directory; Windows does not expose the
        # same operation through os.open, so skip only that directory sync.
        if os.name != "nt":
            directory_fd = os.open(output_dir, os.O_RDONLY)
            try:
                os.fsync(directory_fd)
            finally:
                os.close(directory_fd)
        outputs.append(path)
    plt.show()
    plt.close(fig)
    return outputs


def oriented_percentile(values: Iterable[float], adverse_high: bool = True) -> np.ndarray:
    values = np.asarray(list(values), dtype=float)
    oriented = values if adverse_high else -values
    valid = np.isfinite(oriented)
    output = np.full(oriented.shape, np.nan, dtype=float)
    output[valid] = (rankdata(oriented[valid], method="average") - 0.5) / valid.sum()
    return output


def rank_descending(values: Iterable[float], method: str = "average") -> np.ndarray:
    return rankdata(-np.asarray(list(values), dtype=float), method=method)


def ecdf_landscape_score(ndvi: np.ndarray, ndbi: np.ndarray, mndwi: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    ndvi_p = oriented_percentile(ndvi, adverse_high=False)
    ndbi_p = oriented_percentile(ndbi, adverse_high=True)
    mndwi_p = oriented_percentile(mndwi, adverse_high=False)
    landscape = 0.5 * (0.5 * (ndvi_p + ndbi_p)) + 0.5 * mndwi_p
    return ndvi_p, ndbi_p, mndwi_p, landscape


def load_core_tables(data_root: str | Path) -> dict[str, pd.DataFrame]:
    root = Path(data_root)
    baseline = root / "13_tables" / "baseline_reproduction"
    mapping = {
        "landsat": baseline / "component_epoch_multiscale_landsat.csv",
        "terrain": baseline / "component_multiscale_terrain_hydrology.csv",
        "scores_frozen": baseline / "component_epoch_integrated_scores.csv",
        "climate_epoch_legacy": baseline / "matched_epoch_climate_metrics.csv",
        "climate_annual": root / "13_tables" / "open_meteo_annual_climate_metrics.csv",
        "climate_seasonal": root / "13_tables" / "open_meteo_seasonal_climate_metrics.csv",
        "climate_products": root / "13_tables" / "weather_product_annual_aligned.csv",
        "climate_trends": root / "14_statistics" / "open_meteo_climate_trends_block_bootstrap.csv",
        "weather_lags": root / "13_tables" / "landsat_epoch_weather_lag_summary.csv",
        "model_frozen": root / "13_tables" / "enhanced_spatially_buffered_proxy_model_comparison.csv",
        "model_calibration_frozen": root / "13_tables" / "proxy_model_calibration.csv",
        "model_confusion_frozen": root / "13_tables" / "primary_buffered_proxy_model_confusion_matrix.csv",
        "feature_frame": baseline / "proxy_model_feature_table.csv",
        "rank_block_frozen": root / "14_statistics" / "component_rank_spatial_block_bootstrap.csv",
        "claim_matrix": root / "16_validation" / "claim_evidence_matrix.csv",
        "experiment_status": root / "16_validation" / "experiment_status_matrix.csv",
    }
    tables = {name: pd.read_csv(path) for name, path in mapping.items() if path.exists()}
    return tables


def recompute_fixed_scores(landsat: pd.DataFrame, terrain: pd.DataFrame) -> pd.DataFrame:
    scored_parts: list[pd.DataFrame] = []
    terrain_parts: list[pd.DataFrame] = []
    for radius in RADII:
        local = landsat[landsat["radius_m"].eq(radius)].copy()
        local["ndvi_pressure"] = oriented_percentile(local["ndvi_median"], adverse_high=False)
        local["ndbi_pressure"] = oriented_percentile(local["ndbi_median"], adverse_high=True)
        local["mndwi_pressure"] = oriented_percentile(local["mndwi_median"], adverse_high=False)
        local["surface_cover_pressure"] = local[["ndvi_pressure", "ndbi_pressure"]].mean(axis=1)
        local["landscape_pressure_score"] = local[["surface_cover_pressure", "mndwi_pressure"]].mean(axis=1)
        scored_parts.append(local)

        static = terrain[terrain["radius_m"].eq(radius)].copy()
        static["slope_pressure"] = oriented_percentile(static["slope_p90_deg"], adverse_high=True)
        static["wetness_pressure"] = oriented_percentile(static["twi_proxy_p90"], adverse_high=True)
        static["drainage_proximity_pressure"] = oriented_percentile(
            static["drainage_distance_median_m"], adverse_high=False
        )
        static["terrain_susceptibility_score"] = static[
            ["slope_pressure", "wetness_pressure", "drainage_proximity_pressure"]
        ].mean(axis=1)
        terrain_parts.append(static)

    scores = pd.concat(scored_parts, ignore_index=True)
    terrain_scores = pd.concat(terrain_parts, ignore_index=True)
    merge_columns = [
        "component_id",
        "radius_m",
        "terrain_susceptibility_score",
        "slope_pressure",
        "wetness_pressure",
        "drainage_proximity_pressure",
    ]
    scores = scores.merge(terrain_scores[merge_columns], on=["component_id", "radius_m"], how="left")
    scores["local_priority_score"] = scores[["landscape_pressure_score", "terrain_susceptibility_score"]].mean(axis=1)
    scores["fixed_rank"] = np.nan
    for radius in RADII:
        mask = scores["radius_m"].eq(radius) & scores["epoch_id"].eq("E2024")
        scores.loc[mask, "fixed_rank"] = rank_descending(scores.loc[mask, "local_priority_score"], method="ordinal")
    return scores


def validate_fixed_reproduction(recomputed: pd.DataFrame, frozen: pd.DataFrame, tolerance: float = 2e-7) -> pd.DataFrame:
    keys = ["component_id", "radius_m", "epoch_id"]
    cols = ["landscape_pressure_score", "terrain_susceptibility_score", "local_priority_score"]
    joined = recomputed[keys + cols].merge(frozen[keys + cols], on=keys, suffixes=("_new", "_frozen"))
    rows = []
    for column in cols:
        difference = np.abs(joined[f"{column}_new"] - joined[f"{column}_frozen"])
        rows.append(
            {
                "metric": column,
                "rows": len(joined),
                "max_abs_difference": float(difference.max()),
                "mean_abs_difference": float(difference.mean()),
                "pass": bool(difference.max() <= tolerance),
            }
        )
    return pd.DataFrame(rows)


def load_components(data_root: str | Path, projected: bool = True) -> pd.DataFrame:
    filename = "taxila_components_epsg32643.geojson" if projected else "taxila_components_wgs84.geojson"
    payload = json.loads((Path(data_root) / "06_processed_vectors" / filename).read_text(encoding="utf-8"))
    rows = []
    for feature in payload["features"]:
        if not feature.get("geometry"):
            continue
        properties = feature["properties"]
        x, y = feature["geometry"]["coordinates"]
        rows.append(
            {
                "component_id": properties["component_id"],
                "component_name": properties.get("name", properties.get("component_name")),
                "x": float(x),
                "y": float(y),
                "longitude": float(properties.get("longitude", x if not projected else np.nan)),
                "latitude": float(properties.get("latitude", y if not projected else np.nan)),
            }
        )
    return pd.DataFrame(rows).sort_values("component_id").reset_index(drop=True)


def load_epoch_indices(data_root: str | Path, epoch: str = "E2024") -> tuple[np.ndarray, dict[str, object]]:
    path = Path(data_root) / "05_processed_rasters" / f"{epoch}_oli_like_indices.tif"
    with rasterio.open(path) as source:
        array = source.read().astype(np.float32)
        metadata = {
            "crs": str(source.crs),
            "transform": source.transform,
            "width": source.width,
            "height": source.height,
            "descriptions": source.descriptions,
            "nodata": source.nodata,
        }
    array[array <= -9000] = np.nan
    return np.moveaxis(array, 0, -1), metadata


def load_pressure_grid(data_root: str | Path, epoch: str) -> np.ndarray:
    raster_root = Path(data_root) / "05_processed_rasters"
    binary_path = raster_root / f"{epoch}_relative_landscape_pressure.f32"
    tif_path = raster_root / f"{epoch}_relative_landscape_pressure.tif"
    if binary_path.is_file():
        grid = np.fromfile(binary_path, dtype="<f4").reshape(GRID_HEIGHT, GRID_WIDTH).astype(float)
    elif tif_path.is_file():
        with rasterio.open(tif_path) as source:
            grid = source.read(1).astype(float)
    else:
        raise FileNotFoundError(f"No pressure raster found for {epoch} in {raster_root}")
    grid[grid <= -9000] = np.nan
    return grid


def load_convergence_grid(data_root: str | Path) -> np.ndarray:
    raster_root = Path(data_root) / "05_processed_rasters"
    binary_path = raster_root / "E2004_E2024_spectral_convergence.f32"
    tif_path = raster_root / "E2004_E2024_spectral_convergence.tif"
    if binary_path.is_file():
        grid = np.fromfile(binary_path, dtype="<f4").reshape(GRID_HEIGHT, GRID_WIDTH).astype(float)
    elif tif_path.is_file():
        with rasterio.open(tif_path) as source:
            grid = source.read(1).astype(float)
    else:
        raise FileNotFoundError(f"No convergence raster found in {raster_root}")
    grid[grid <= -9000] = np.nan
    return grid


def grid_slice_utm(x: float, y: float, radius_m: float) -> tuple[slice, slice, np.ndarray, np.ndarray]:
    column = int((x - GRID_XMIN) / GRID_CELL)
    row = int((GRID_YMAX - y) / GRID_CELL)
    half = int(math.ceil(radius_m / GRID_CELL)) + 2
    row0, row1 = max(0, row - half), min(GRID_HEIGHT, row + half + 1)
    col0, col1 = max(0, column - half), min(GRID_WIDTH, column + half + 1)
    xs = GRID_XMIN + (np.arange(col0, col1) + 0.5) * GRID_CELL
    ys = GRID_YMAX - (np.arange(row0, row1) + 0.5) * GRID_CELL
    return slice(row0, row1), slice(col0, col1), xs, ys


def extract_spectral_medians(indices: np.ndarray, points: pd.DataFrame, radius_m: float = 500) -> np.ndarray:
    output = np.empty((len(points), 3), dtype=float)
    for index, point in enumerate(points.itertuples(index=False)):
        rows, cols, xs, ys = grid_slice_utm(float(point.x), float(point.y), radius_m)
        xx, yy = np.meshgrid(xs, ys)
        use = (xx - float(point.x)) ** 2 + (yy - float(point.y)) ** 2 <= radius_m**2
        values = indices[rows, cols, :3]
        use &= np.all(np.isfinite(values), axis=2)
        output[index] = np.nanmedian(values[use], axis=0)
    return output


def calculate_d8_flow(dem: np.ndarray, cell_y_m: float, cell_x_m: float) -> np.ndarray:
    height, width = dem.shape
    count = height * width
    best_slope = np.zeros_like(dem, dtype=np.float32)
    receiver = np.full((height, width), -1, dtype=np.int64)
    flat_indices = np.arange(count, dtype=np.int64).reshape(height, width)
    for dr, dc in [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]:
        source_rows = slice(max(0, -dr), min(height, height - dr))
        source_cols = slice(max(0, -dc), min(width, width - dc))
        target_rows = slice(max(0, dr), min(height, height + dr))
        target_cols = slice(max(0, dc), min(width, width + dc))
        distance = math.hypot(cell_y_m * dr, cell_x_m * dc)
        slope = (dem[source_rows, source_cols] - dem[target_rows, target_cols]) / distance
        update = slope > best_slope[source_rows, source_cols]
        best_slope[source_rows, source_cols][update] = slope[update]
        receiver[source_rows, source_cols][update] = flat_indices[target_rows, target_cols][update]
    receiver_flat = receiver.ravel()
    valid_receiver = receiver_flat >= 0
    indegree = np.bincount(receiver_flat[valid_receiver], minlength=count).astype(np.int32)
    accumulation = np.ones(count, dtype=float)
    queue = deque(np.where(indegree == 0)[0].tolist())
    processed = 0
    while queue:
        node = queue.popleft()
        processed += 1
        target = receiver_flat[node]
        if target >= 0:
            accumulation[target] += accumulation[node]
            indegree[target] -= 1
            if indegree[target] == 0:
                queue.append(int(target))
    if processed != count:
        raise RuntimeError(f"D8 graph is cyclic: processed {processed}/{count}")
    return accumulation.reshape(height, width)


def build_terrain_context(data_root: str | Path, components_wgs84: pd.DataFrame) -> dict[str, np.ndarray | float]:
    hgt_path = Path(data_root) / "02_raw_data" / "N33E072.hgt.gz"
    with gzip.open(hgt_path, "rb") as stream:
        full = np.frombuffer(stream.read(), dtype=">i2").reshape(3601, 3601).astype(np.float32)
    full[full <= -32768] = np.nan
    # Frozen terrain workflow crop margin; changing it changes D8 edge conditions.
    margin = 0.035
    lat_min = max(33.0, float(components_wgs84["latitude"].min()) - margin)
    lat_max = min(34.0, float(components_wgs84["latitude"].max()) + margin)
    lon_min = max(72.0, float(components_wgs84["longitude"].min()) - margin)
    lon_max = min(73.0, float(components_wgs84["longitude"].max()) + margin)
    row0 = max(0, int(math.floor((34.0 - lat_max) * 3600)))
    row1 = min(3601, int(math.ceil((34.0 - lat_min) * 3600)) + 1)
    col0 = max(0, int(math.floor((lon_min - 72.0) * 3600)))
    col1 = min(3601, int(math.ceil((lon_max - 72.0) * 3600)) + 1)
    dem = full[row0:row1, col0:col1]
    latitudes = 34.0 - np.arange(row0, row1) / 3600.0
    longitudes = 72.0 + np.arange(col0, col1) / 3600.0
    mean_latitude = float(np.mean(latitudes))
    cell_y_m = 111_132.0 / 3600.0
    cell_x_m = 111_320.0 * math.cos(math.radians(mean_latitude)) / 3600.0
    gradient_y, gradient_x = np.gradient(dem, cell_y_m, cell_x_m)
    slope = np.degrees(np.arctan(np.hypot(gradient_x, gradient_y))).astype(np.float32)
    flow = calculate_d8_flow(dem, cell_y_m, cell_x_m)
    slope_radians = np.radians(np.clip(slope, 0.05, None))
    wetness = (
        np.log((flow + 1.0) * math.sqrt(cell_x_m * cell_y_m)) - np.log(np.tan(slope_radians) + 0.01)
    ).astype(np.float32)
    threshold = float(np.nanquantile(flow, 0.99))
    drainage = flow >= threshold
    drainage_distance = ndimage.distance_transform_edt(~drainage, sampling=(cell_y_m, cell_x_m)).astype(np.float32)
    return {
        "dem": dem,
        "slope": slope,
        "flow": flow,
        "wetness": wetness,
        "drainage_distance": drainage_distance,
        "drainage_mask": drainage,
        "latitudes": latitudes,
        "longitudes": longitudes,
        "cell_y_m": cell_y_m,
        "cell_x_m": cell_x_m,
        "drainage_threshold": threshold,
    }


def extract_terrain_metrics(
    terrain_context: dict[str, np.ndarray | float], points_wgs84: pd.DataFrame, radius_m: float = 500
) -> np.ndarray:
    latitudes = np.asarray(terrain_context["latitudes"])
    longitudes = np.asarray(terrain_context["longitudes"])
    output = np.empty((len(points_wgs84), 3), dtype=float)
    mean_lat = float(np.mean(latitudes))
    cell_y = float(terrain_context["cell_y_m"])
    cell_x = float(terrain_context["cell_x_m"])
    half_rows = int(math.ceil(radius_m / cell_y)) + 2
    half_cols = int(math.ceil(radius_m / cell_x)) + 2
    for index, point in enumerate(points_wgs84.itertuples(index=False)):
        row = int(np.argmin(np.abs(latitudes - float(point.latitude))))
        col = int(np.argmin(np.abs(longitudes - float(point.longitude))))
        row0, row1 = max(0, row - half_rows), min(len(latitudes), row + half_rows + 1)
        col0, col1 = max(0, col - half_cols), min(len(longitudes), col + half_cols + 1)
        local_lat = latitudes[row0:row1]
        local_lon = longitudes[col0:col1]
        lon_grid, lat_grid = np.meshgrid(local_lon, local_lat)
        dx = (lon_grid - float(point.longitude)) * 111_320.0 * math.cos(math.radians(float(point.latitude)))
        dy = (lat_grid - float(point.latitude)) * 111_132.0
        use = dx**2 + dy**2 <= radius_m**2
        slope = np.asarray(terrain_context["slope"])[row0:row1, col0:col1]
        wetness = np.asarray(terrain_context["wetness"])[row0:row1, col0:col1]
        distance = np.asarray(terrain_context["drainage_distance"])[row0:row1, col0:col1]
        output[index] = [
            np.nanpercentile(slope[use], 90),
            np.nanpercentile(wetness[use], 90),
            np.nanmedian(distance[use]),
        ]
    return output


def terrain_pressures(metrics: np.ndarray) -> np.ndarray:
    return np.column_stack(
        [
            oriented_percentile(metrics[:, 0], adverse_high=True),
            oriented_percentile(metrics[:, 1], adverse_high=True),
            oriented_percentile(metrics[:, 2], adverse_high=False),
        ]
    )


def pooled_landscape_from_replacement(
    baseline_landsat_500m: pd.DataFrame, component_ids: list[str], replacement_medians: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Convert one perturbed E2024 inventory into endpoint empirical pressures.

    The frozen fixed score uses all five epoch-component observations for its
    deterministic ECDF. Uncertainty draws instead rerank the 17 simultaneously
    perturbed endpoint component summaries, as declared in the shared-block and
    point-location experiments. ``baseline_landsat_500m`` is retained in the
    signature to make the frozen input dependency explicit and to preserve the
    notebook API.
    """
    if len(component_ids) != len(replacement_medians):
        raise ValueError("Replacement medians must contain one row per mapped component.")
    ndvi_p, ndbi_p, mndwi_p, landscape = ecdf_landscape_score(
        replacement_medians[:, 0], replacement_medians[:, 1], replacement_medians[:, 2]
    )
    factors = np.column_stack([ndvi_p, ndbi_p, mndwi_p])
    return landscape, factors


def displace_points(
    projected: pd.DataFrame, geographic: pd.DataFrame, maximum_distance_m: float, rng: np.random.Generator
) -> tuple[pd.DataFrame, pd.DataFrame]:
    angles = rng.uniform(0, 2 * np.pi, len(projected))
    radii = maximum_distance_m * np.sqrt(rng.uniform(0, 1, len(projected)))
    dx = radii * np.cos(angles)
    dy = radii * np.sin(angles)
    shifted_projected = projected.copy()
    shifted_projected["x"] = shifted_projected["x"].to_numpy() + dx
    shifted_projected["y"] = shifted_projected["y"].to_numpy() + dy
    shifted_geographic = geographic.copy()
    shifted_geographic["longitude"] = shifted_geographic["longitude"].to_numpy() + dx / (
        111_320.0 * np.cos(np.radians(shifted_geographic["latitude"].to_numpy()))
    )
    shifted_geographic["latitude"] = shifted_geographic["latitude"].to_numpy() + dy / 111_132.0
    return shifted_projected, shifted_geographic


def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[valid]
    weights = weights[valid]
    if not len(values):
        return np.nan
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cutoff = 0.5 * weights.sum()
    return float(values[np.searchsorted(np.cumsum(weights), cutoff, side="left")])


def block_summary_for_points(
    indices: np.ndarray, points: pd.DataFrame, radius_m: float, block_size_m: int
) -> list[dict[str, np.ndarray]]:
    cells_per_block = max(1, int(round(block_size_m / GRID_CELL)))
    blocks_across = math.ceil(GRID_WIDTH / cells_per_block)
    summaries: list[dict[str, np.ndarray]] = []
    for point in points.itertuples(index=False):
        rows, cols, xs, ys = grid_slice_utm(float(point.x), float(point.y), radius_m)
        xx, yy = np.meshgrid(xs, ys)
        local_values = indices[rows, cols, :3]
        use = (xx - float(point.x)) ** 2 + (yy - float(point.y)) ** 2 <= radius_m**2
        use &= np.all(np.isfinite(local_values), axis=2)
        local_rows, local_cols = np.meshgrid(
            np.arange(rows.start, rows.stop), np.arange(cols.start, cols.stop), indexing="ij"
        )
        local_block_ids = (local_rows // cells_per_block) * blocks_across + local_cols // cells_per_block
        ids = local_block_ids[use]
        values = local_values[use]
        unique = np.unique(ids)
        medians = np.vstack([np.nanmedian(values[ids == block], axis=0) for block in unique])
        counts = np.asarray([np.sum(ids == block) for block in unique], dtype=float)
        summaries.append({"block_ids": unique.astype(int), "medians": medians, "counts": counts})
    return summaries


def sample_shared_block_medians(
    summaries: list[dict[str, np.ndarray]], rng: np.random.Generator, maximum_block_id: int | None = None
) -> np.ndarray:
    if maximum_block_id is None:
        maximum_block_id = max(int(summary["block_ids"].max()) for summary in summaries)
    shared_weights = rng.exponential(scale=1.0, size=maximum_block_id + 1)
    output = np.empty((len(summaries), 3), dtype=float)
    for component_index, summary in enumerate(summaries):
        weights = shared_weights[summary["block_ids"]] * summary["counts"]
        output[component_index] = [
            weighted_median(summary["medians"][:, factor], weights) for factor in range(3)
        ]
    return output


def summarise_rank_draws(
    component_ids: list[str], component_names: list[str], scores: np.ndarray, ranks: np.ndarray, scope: str
) -> pd.DataFrame:
    rows = []
    for index, component_id in enumerate(component_ids):
        rows.append(
            {
                "component_id": component_id,
                "component_name": component_names[index],
                "score_median": float(np.median(scores[:, index])),
                "score_ci_low_95": float(np.percentile(scores[:, index], 2.5)),
                "score_ci_high_95": float(np.percentile(scores[:, index], 97.5)),
                "rank_median": float(np.median(ranks[:, index])),
                "rank_ci_low_95": float(np.percentile(ranks[:, index], 2.5)),
                "rank_ci_high_95": float(np.percentile(ranks[:, index], 97.5)),
                "probability_top_3": float(np.mean(ranks[:, index] <= 3)),
                "probability_top_5": float(np.mean(ranks[:, index] <= 5)),
                "draws": int(len(ranks)),
                "uncertainty_scope": scope,
            }
        )
    return pd.DataFrame(rows).sort_values(["rank_median", "probability_top_3"], ascending=[True, False])


def fixed_factor_frame(scores: pd.DataFrame, radius_m: int = 500) -> pd.DataFrame:
    columns = [
        "component_id",
        "component_name",
        "ndvi_pressure",
        "ndbi_pressure",
        "mndwi_pressure",
        "slope_pressure",
        "wetness_pressure",
        "drainage_proximity_pressure",
        "surface_cover_pressure",
        "landscape_pressure_score",
        "terrain_susceptibility_score",
        "local_priority_score",
    ]
    return (
        scores[scores["radius_m"].eq(radius_m) & scores["epoch_id"].eq("E2024")][columns]
        .sort_values("component_id")
        .reset_index(drop=True)
    )


def terrain_formulations(terrain_factor_pressures: np.ndarray) -> dict[str, np.ndarray]:
    factors = np.asarray(terrain_factor_pressures, dtype=float)
    correlation = np.abs(pd.DataFrame(factors).corr(method="spearman").to_numpy())
    inverse = 1.0 / np.maximum(correlation.sum(axis=1), 1e-12)
    correlation_weights = inverse / inverse.sum()
    return {
        "equal_factors": factors.mean(axis=1),
        "correlation_adjusted": factors @ correlation_weights,
        "reduced_slope_drainage": factors[:, [0, 2]].mean(axis=1),
    }


def weight_concentration_experiment(
    factors: pd.DataFrame,
    alphas: Iterable[float] = (0.5, 1.0, 2.0),
    draws: int = 50_000,
    seed: int = SEED,
) -> tuple[pd.DataFrame, dict[float, np.ndarray]]:
    component_ids = factors["component_id"].tolist()
    component_names = factors["component_name"].tolist()
    landscape = factors["landscape_pressure_score"].to_numpy(dtype=float)
    terrain = factors["terrain_susceptibility_score"].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    summaries = []
    rank_draws: dict[float, np.ndarray] = {}
    for alpha in alphas:
        weights = rng.beta(alpha, alpha, size=draws)
        score = weights[:, None] * landscape + (1.0 - weights[:, None]) * terrain
        order = np.argsort(-score, axis=1)
        ranks = np.empty_like(order, dtype=np.int16)
        row_indices = np.arange(draws)[:, None]
        ranks[row_indices, order] = np.arange(1, len(factors) + 1, dtype=np.int16)
        rank_draws[float(alpha)] = ranks
        for index, component_id in enumerate(component_ids):
            summaries.append(
                {
                    "alpha": float(alpha),
                    "component_id": component_id,
                    "component_name": component_names[index],
                    "rank_median": float(np.median(ranks[:, index])),
                    "rank_ci_low_95": float(np.percentile(ranks[:, index], 2.5)),
                    "rank_ci_high_95": float(np.percentile(ranks[:, index], 97.5)),
                    "probability_top_3": float(np.mean(ranks[:, index] <= 3)),
                    "probability_top_5": float(np.mean(ranks[:, index] <= 5)),
                    "draws": draws,
                }
            )
    return pd.DataFrame(summaries), rank_draws


def shared_block_size_experiment(
    indices: np.ndarray,
    projected_points: pd.DataFrame,
    baseline_landsat_500m: pd.DataFrame,
    fixed_terrain_score: np.ndarray,
    block_sizes_m: Iterable[int] = (90, 150, 300),
    draws: int = 2_000,
    seed: int = SEED,
) -> tuple[pd.DataFrame, dict[int, np.ndarray]]:
    component_ids = projected_points["component_id"].tolist()
    component_names = projected_points["component_name"].tolist()
    rng = np.random.default_rng(seed)
    summary_rows = []
    raw_ranks: dict[int, np.ndarray] = {}
    for block_size in block_sizes_m:
        blocks = block_summary_for_points(indices, projected_points, 500, int(block_size))
        scores = np.empty((draws, len(projected_points)), dtype=float)
        ranks = np.empty_like(scores, dtype=np.int16)
        for draw in range(draws):
            medians = sample_shared_block_medians(blocks, rng)
            landscape, _ = pooled_landscape_from_replacement(baseline_landsat_500m, component_ids, medians)
            scores[draw] = 0.5 * landscape + 0.5 * fixed_terrain_score
            ranks[draw] = rank_descending(scores[draw], method="ordinal").astype(np.int16)
        raw_ranks[int(block_size)] = ranks
        for index, component_id in enumerate(component_ids):
            summary_rows.append(
                {
                    "block_size_m": int(block_size),
                    "component_id": component_id,
                    "component_name": component_names[index],
                    "rank_median": float(np.median(ranks[:, index])),
                    "rank_ci_low_95": float(np.percentile(ranks[:, index], 2.5)),
                    "rank_ci_high_95": float(np.percentile(ranks[:, index], 97.5)),
                    "probability_top_3": float(np.mean(ranks[:, index] <= 3)),
                    "probability_top_5": float(np.mean(ranks[:, index] <= 5)),
                    "draws": draws,
                    "method": "shared support-weighted Bayesian block bootstrap",
                }
            )
    return pd.DataFrame(summary_rows), raw_ranks


def point_displacement_experiment(
    indices: np.ndarray,
    projected_points: pd.DataFrame,
    geographic_points: pd.DataFrame,
    terrain_context: dict[str, np.ndarray | float],
    baseline_landsat_500m: pd.DataFrame,
    maximum_distances_m: Iterable[int] = (0, 30, 60, 120),
    draws: int = 2_000,
    seed: int = SEED,
) -> tuple[pd.DataFrame, dict[int, np.ndarray]]:
    component_ids = projected_points["component_id"].tolist()
    component_names = projected_points["component_name"].tolist()
    rng = np.random.default_rng(seed)
    summary_rows = []
    raw_ranks: dict[int, np.ndarray] = {}
    for maximum_distance in maximum_distances_m:
        scores = np.empty((draws, len(projected_points)), dtype=float)
        ranks = np.empty_like(scores, dtype=np.int16)
        effective_draws = 1 if maximum_distance == 0 else draws
        for draw in range(effective_draws):
            if maximum_distance == 0:
                projected_draw, geographic_draw = projected_points, geographic_points
            else:
                projected_draw, geographic_draw = displace_points(
                    projected_points, geographic_points, float(maximum_distance), rng
                )
            spectral = extract_spectral_medians(indices, projected_draw, radius_m=500)
            landscape, _ = pooled_landscape_from_replacement(baseline_landsat_500m, component_ids, spectral)
            terrain_metrics = extract_terrain_metrics(terrain_context, geographic_draw, radius_m=500)
            terrain = terrain_pressures(terrain_metrics).mean(axis=1)
            scores[draw] = 0.5 * landscape + 0.5 * terrain
            ranks[draw] = rank_descending(scores[draw], method="ordinal").astype(np.int16)
        if effective_draws == 1 and draws > 1:
            scores[1:] = scores[0]
            ranks[1:] = ranks[0]
        raw_ranks[int(maximum_distance)] = ranks
        for index, component_id in enumerate(component_ids):
            summary_rows.append(
                {
                    "maximum_displacement_m": int(maximum_distance),
                    "component_id": component_id,
                    "component_name": component_names[index],
                    "rank_median": float(np.median(ranks[:, index])),
                    "rank_ci_low_95": float(np.percentile(ranks[:, index], 2.5)),
                    "rank_ci_high_95": float(np.percentile(ranks[:, index], 97.5)),
                    "probability_top_3": float(np.mean(ranks[:, index] <= 3)),
                    "probability_top_5": float(np.mean(ranks[:, index] <= 5)),
                    "draws": draws,
                    "terrain_reextracted": True,
                }
            )
    return pd.DataFrame(summary_rows), raw_ranks


def structural_scenarios(scores: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for radius in RADII:
        factors = fixed_factor_frame(scores, radius)
        terrain_factor_array = factors[["slope_pressure", "wetness_pressure", "drainage_proximity_pressure"]].to_numpy()
        terrain_models = terrain_formulations(terrain_factor_array)
        landscape = factors["landscape_pressure_score"].to_numpy()
        for model_name, terrain in terrain_models.items():
            for landscape_weight in (0.25, 0.50, 0.75):
                score = landscape_weight * landscape + (1.0 - landscape_weight) * terrain
                rank = rank_descending(score, method="ordinal").astype(int)
                for index, record in factors.iterrows():
                    rows.append(
                        {
                            "component_id": record["component_id"],
                            "component_name": record["component_name"],
                            "radius_m": radius,
                            "terrain_model": model_name,
                            "landscape_weight": landscape_weight,
                            "terrain_weight": 1.0 - landscape_weight,
                            "score": score[index],
                            "rank": rank[index],
                            "top3": rank[index] <= 3,
                            "top5": rank[index] <= 5,
                        }
                    )
    scenarios = pd.DataFrame(rows)
    stability = (
        scenarios.groupby(["component_id", "component_name"])
        .agg(
            scenarios=("rank", "size"),
            median_rank=("rank", "median"),
            rank_min=("rank", "min"),
            rank_max=("rank", "max"),
            top3_frequency=("top3", "mean"),
            top5_frequency=("top5", "mean"),
        )
        .reset_index()
    )
    stability["inspection_tier"] = np.select(
        [stability["top3_frequency"].ge(0.50), stability["top5_frequency"].ge(0.50)],
        ["Tier 1: robust high priority", "Tier 2: conditional high priority"],
        default="Tier 3: routine/uncertain priority",
    )
    return scenarios, stability.sort_values(["inspection_tier", "median_rank"])


def joint_uncertainty_experiment(
    indices: np.ndarray,
    projected_points: pd.DataFrame,
    geographic_points: pd.DataFrame,
    terrain_context: dict[str, np.ndarray | float],
    baseline_landsat_500m: pd.DataFrame,
    spatial_states: int = 500,
    decision_draws_per_state: int = 10,
    seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    component_ids = projected_points["component_id"].tolist()
    component_names = projected_points["component_name"].tolist()
    rng = np.random.default_rng(seed)
    score_rows: list[dict[str, object]] = []
    rank_matrix = np.empty((spatial_states * decision_draws_per_state, len(projected_points)), dtype=np.int16)
    score_matrix = np.empty_like(rank_matrix, dtype=float)
    draw_index = 0
    for state in range(spatial_states):
        block_size = int(rng.choice([90, 150, 300]))
        # Each component is displaced independently in a uniform-area disc of radius 120 m.
        maximum_displacement = 120.0
        projected_draw, geographic_draw = displace_points(
            projected_points, geographic_points, maximum_displacement, rng
        )
        block_summaries = block_summary_for_points(indices, projected_draw, 500, block_size)
        spectral = sample_shared_block_medians(block_summaries, rng)
        landscape, landscape_factors = pooled_landscape_from_replacement(
            baseline_landsat_500m, component_ids, spectral
        )
        terrain_metrics = extract_terrain_metrics(terrain_context, geographic_draw, radius_m=500)
        terrain_factor_array = terrain_pressures(terrain_metrics)
        terrain_models = terrain_formulations(terrain_factor_array)
        model_names = list(terrain_models)
        for decision in range(decision_draws_per_state):
            landscape_weight = float(rng.beta(1.0, 1.0))
            terrain_model = str(rng.choice(model_names))
            terrain = terrain_models[terrain_model]
            score = landscape_weight * landscape + (1.0 - landscape_weight) * terrain
            ranks = rank_descending(score, method="ordinal").astype(np.int16)
            score_matrix[draw_index] = score
            rank_matrix[draw_index] = ranks
            for component_index, component_id in enumerate(component_ids):
                score_rows.append(
                    {
                        "spatial_state_id": state,
                        "decision_setting_id": decision,
                        "joint_draw_id": draw_index,
                        "block_size_m": block_size,
                        "maximum_displacement_m": maximum_displacement,
                        "landscape_weight": landscape_weight,
                        "terrain_model": terrain_model,
                        "component_id": component_id,
                        "component_name": component_names[component_index],
                        "score": float(score[component_index]),
                        "rank": int(ranks[component_index]),
                        "top3": bool(ranks[component_index] <= 3),
                        "top5": bool(ranks[component_index] <= 5),
                    }
                )
            draw_index += 1
    summary = summarise_rank_draws(
        component_ids,
        component_names,
        score_matrix,
        rank_matrix,
        scope=(
            f"{spatial_states} independent spatial states crossed with "
            f"{decision_draws_per_state} decision settings; 500 m primary support"
        ),
    )
    summary["spatial_states"] = spatial_states
    summary["decision_draws_per_state"] = decision_draws_per_state
    summary["joint_score_vectors"] = spatial_states * decision_draws_per_state
    return pd.DataFrame(score_rows), summary


def support_overlap_table(projected_points: pd.DataFrame, radii: Iterable[int] = RADII) -> pd.DataFrame:
    rows = []
    for radius in radii:
        circle_area = math.pi * radius**2
        for left in range(len(projected_points)):
            for right in range(left + 1, len(projected_points)):
                a = projected_points.iloc[left]
                b = projected_points.iloc[right]
                distance = math.hypot(float(a["x"] - b["x"]), float(a["y"] - b["y"]))
                if distance >= 2 * radius:
                    intersection = 0.0
                else:
                    intersection = 2 * radius**2 * math.acos(distance / (2 * radius)) - 0.5 * distance * math.sqrt(
                        max(4 * radius**2 - distance**2, 0.0)
                    )
                jaccard = intersection / (2 * circle_area - intersection) if intersection > 0 else 0.0
                rows.append(
                    {
                        "radius_m": int(radius),
                        "component_a": a["component_name"],
                        "component_b": b["component_name"],
                        "distance_m": distance,
                        "intersection_area_m2": intersection,
                        "jaccard_overlap": jaccard,
                        "flag_cluster_aware": bool(radius == 500 and jaccard >= 0.25),
                    }
                )
    return pd.DataFrame(rows).sort_values(["radius_m", "jaccard_overlap"], ascending=[True, False])


def ablation_experiment(factors: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    reference_score = factors["local_priority_score"].to_numpy()
    reference_rank = rank_descending(reference_score, method="average")
    spectral = factors[["ndvi_pressure", "ndbi_pressure", "mndwi_pressure"]].copy()
    terrain = factors[["slope_pressure", "wetness_pressure", "drainage_proximity_pressure"]].copy()
    alternatives: dict[str, np.ndarray] = {
        "Full hierarchy": reference_score,
        "Landscape only": factors["landscape_pressure_score"].to_numpy(),
        "Terrain only": factors["terrain_susceptibility_score"].to_numpy(),
        "Without NDVI": 0.5 * factors[["ndbi_pressure", "mndwi_pressure"]].mean(axis=1).to_numpy()
        + 0.5 * factors["terrain_susceptibility_score"].to_numpy(),
        "Without NDBI": 0.5 * factors[["ndvi_pressure", "mndwi_pressure"]].mean(axis=1).to_numpy()
        + 0.5 * factors["terrain_susceptibility_score"].to_numpy(),
        "Without MNDWI": 0.5 * factors[["ndvi_pressure", "ndbi_pressure"]].mean(axis=1).to_numpy()
        + 0.5 * factors["terrain_susceptibility_score"].to_numpy(),
        "Without slope": 0.5 * factors["landscape_pressure_score"].to_numpy()
        + 0.5 * terrain[["wetness_pressure", "drainage_proximity_pressure"]].mean(axis=1).to_numpy(),
        "Without wetness": 0.5 * factors["landscape_pressure_score"].to_numpy()
        + 0.5 * terrain[["slope_pressure", "drainage_proximity_pressure"]].mean(axis=1).to_numpy(),
        "Without drainage": 0.5 * factors["landscape_pressure_score"].to_numpy()
        + 0.5 * terrain[["slope_pressure", "wetness_pressure"]].mean(axis=1).to_numpy(),
    }
    summary_rows = []
    rank_table = factors[["component_id", "component_name"]].copy()
    reference_top5 = set(np.where(reference_rank <= 5)[0])
    for name, score in alternatives.items():
        rank = rank_descending(score, method="average")
        rank_table[name] = rank
        summary_rows.append(
            {
                "scenario": name,
                "spearman_rho_with_reference": float(spearmanr(reference_rank, rank).statistic),
                "kendall_tau_with_reference": float(kendalltau(reference_rank, rank).statistic),
                "top5_overlap": len(reference_top5 & set(np.where(rank <= 5)[0])),
                "mean_absolute_rank_shift": float(np.mean(np.abs(reference_rank - rank))),
                "maximum_absolute_rank_shift": float(np.max(np.abs(reference_rank - rank))),
                "rank_1_component": factors.iloc[int(np.argmin(rank))]["component_name"],
            }
        )
    return pd.DataFrame(summary_rows), rank_table


def baseline_comparison(factors: pd.DataFrame, raw_landsat_500: pd.DataFrame, raw_terrain_500: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    factor_columns = [
        "ndvi_pressure",
        "ndbi_pressure",
        "mndwi_pressure",
        "slope_pressure",
        "wetness_pressure",
        "drainage_proximity_pressure",
    ]
    matrix = factors[factor_columns].to_numpy(dtype=float)
    reference = factors["local_priority_score"].to_numpy(dtype=float)
    reference_rank = rank_descending(reference, method="average")
    correlation = np.corrcoef(matrix, rowvar=False)
    critic_information = np.std(matrix, axis=0, ddof=1) * np.sum(1 - correlation, axis=1)
    critic_weights = critic_information / critic_information.sum()
    shifted = matrix - np.nanmin(matrix, axis=0) + 1e-12
    proportions = shifted / shifted.sum(axis=0, keepdims=True)
    entropy = -np.sum(proportions * np.log(proportions), axis=0) / np.log(len(matrix))
    entropy_weights = (1 - entropy) / np.sum(1 - entropy)

    raw_spectral = raw_landsat_500[raw_landsat_500["epoch_id"].eq("E2024")].sort_values("component_id")
    raw_terrain = raw_terrain_500.sort_values("component_id")
    raw_oriented = np.column_stack(
        [
            -raw_spectral["ndvi_median"].to_numpy(),
            raw_spectral["ndbi_median"].to_numpy(),
            -raw_spectral["mndwi_median"].to_numpy(),
            raw_terrain["slope_p90_deg"].to_numpy(),
            raw_terrain["twi_proxy_p90"].to_numpy(),
            -raw_terrain["drainage_distance_median_m"].to_numpy(),
        ]
    )
    median = np.median(raw_oriented, axis=0)
    mad = np.median(np.abs(raw_oriented - median), axis=0)
    robust_z = (raw_oriented - median) / np.where(mad > 0, 1.4826 * mad, 1.0)
    robust_score = expit(robust_z).mean(axis=1)
    standard = StandardScaler().fit_transform(matrix)
    pca = PCA(n_components=1, random_state=SEED)
    pc_score = pca.fit_transform(standard).ravel()
    if np.mean([spearmanr(pc_score, matrix[:, index]).statistic for index in range(matrix.shape[1])]) < 0:
        pc_score *= -1

    alternatives = {
        "Hierarchical reference": reference,
        "Ungrouped equal-factor mean": matrix.mean(axis=1),
        "Landscape only": factors["landscape_pressure_score"].to_numpy(),
        "Terrain only": factors["terrain_susceptibility_score"].to_numpy(),
        "CRITIC weighting": matrix @ critic_weights,
        "Entropy weighting": matrix @ entropy_weights,
        "Robust-z logistic": robust_score,
        "Oriented first principal component": pc_score,
    }
    reference_top5 = set(np.where(reference_rank <= 5)[0])
    summary_rows = []
    rank_table = factors[["component_id", "component_name"]].copy()
    for name, score in alternatives.items():
        rank = rank_descending(score, method="average")
        rank_table[name] = rank
        summary_rows.append(
            {
                "baseline": name,
                "spearman_rho_with_reference": float(spearmanr(reference_rank, rank).statistic),
                "kendall_tau_with_reference": float(kendalltau(reference_rank, rank).statistic),
                "top5_overlap": len(reference_top5 & set(np.where(rank <= 5)[0])),
                "mean_absolute_rank_shift": float(np.mean(np.abs(reference_rank - rank))),
                "maximum_absolute_rank_shift": float(np.max(np.abs(reference_rank - rank))),
                "rank_1_component": factors.iloc[int(np.argmin(rank))]["component_name"],
            }
        )
    return pd.DataFrame(summary_rows), rank_table


def benjamini_hochberg(p_values: Iterable[float]) -> np.ndarray:
    values = np.asarray(list(p_values), dtype=float)
    order = np.argsort(values)
    adjusted = np.empty_like(values)
    running = 1.0
    for reverse_index in range(len(values) - 1, -1, -1):
        original_index = order[reverse_index]
        candidate = values[original_index] * len(values) / (reverse_index + 1)
        running = min(running, candidate)
        adjusted[original_index] = min(running, 1.0)
    return adjusted


def moving_block_trend_experiment(
    annual: pd.DataFrame,
    metrics: Iterable[str] = (
        "precipitation_sum_mm",
        "rx1day_mm",
        "rx5day_mm",
        "heavy_precipitation_days",
        "consecutive_dry_days",
        "temperature_mean_c",
        "annual_or_period_max_tmax_c",
        "hot_days",
        "heatwave_max_days",
        "relative_humidity_mean_percent",
        "soil_moisture_mean_m3m3",
        "et0_sum_mm",
    ),
    block_length_years: int = 5,
    draws: int = 2_000,
    seed: int = SEED,
) -> pd.DataFrame:
    """Residual moving-block intervals and a centred-null slope test (annual family)."""
    rng = np.random.default_rng(seed)
    rows = []
    for metric in metrics:
        local = annual[["year", metric]].dropna().sort_values("year")
        years = local["year"].to_numpy(dtype=float)
        values = local[metric].to_numpy(dtype=float)
        slope, intercept, scipy_low, scipy_high = theilslopes(values, years, alpha=0.95)
        fitted = np.median(values - slope * years) + slope * years
        residuals = values - fitted
        slopes = np.empty(draws, dtype=float)
        null_slopes = np.empty(draws, dtype=float)
        n = len(values)
        valid_starts = np.arange(0, n - block_length_years + 1)
        for draw in range(draws):
            sampled = []
            while len(sampled) < n:
                start = int(rng.choice(valid_starts))
                sampled.extend(residuals[start : start + block_length_years])
            sample = np.asarray(sampled[:n], dtype=float)
            slopes[draw] = theilslopes(fitted + sample, years)[0]
            null_slopes[draw] = theilslopes(np.median(values) + sample, years)[0]
        two_sided_p = float((1 + np.sum(np.abs(null_slopes) >= abs(slope))) / (draws + 1))
        lag1 = pd.Series(residuals).autocorr(lag=1)
        effective_n = float(np.clip(n * (1 - lag1) / (1 + lag1), 2, n)) if np.isfinite(lag1) else n
        rows.append(
            {
                "period": "annual",
                "metric": metric,
                "n": n,
                "effective_n_lag1": effective_n,
                "theil_sen_slope_per_year": slope,
                "theil_sen_intercept": intercept,
                "scipy_slope_ci_low_95": scipy_low,
                "scipy_slope_ci_high_95": scipy_high,
                "moving_block_slope_ci_low_95": float(np.percentile(slopes, 2.5)),
                "moving_block_slope_ci_high_95": float(np.percentile(slopes, 97.5)),
                "moving_block_p_two_sided": max(two_sided_p, 1 / (draws + 1)),
                "bootstrap_draws": draws,
                "block_length_years": block_length_years,
            }
        )
    output = pd.DataFrame(rows)
    output["fdr_adjusted_block_p"] = benjamini_hochberg(output["moving_block_p_two_sided"])
    output["status"] = "INFERENTIAL_WITH_BLOCK_BOOTSTRAP"
    return output


def top_label_ece(y_true: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> tuple[float, pd.DataFrame]:
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correctness = (prediction == y_true).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    rows = []
    for bin_index, (left, right) in enumerate(zip(edges[:-1], edges[1:])):
        use = (confidence >= left) & ((confidence < right) | ((bin_index == bins - 1) & (confidence <= right)))
        count = int(use.sum())
        accuracy = float(correctness[use].mean()) if count else np.nan
        mean_confidence = float(confidence[use].mean()) if count else np.nan
        if count:
            ece += count / len(y_true) * abs(accuracy - mean_confidence)
        rows.append(
            {
                "bin_left": left,
                "bin_right": right,
                "count": count,
                "accuracy": accuracy,
                "mean_confidence": mean_confidence,
            }
        )
    result = pd.DataFrame(rows)
    result["ece"] = ece
    return float(ece), result


def proxy_metrics(y_true: np.ndarray, prediction: np.ndarray, probabilities: np.ndarray) -> dict[str, float]:
    classes = np.arange(probabilities.shape[1])
    binary = label_binarize(y_true, classes=classes)
    if binary.shape[1] != probabilities.shape[1]:
        complete = np.zeros_like(probabilities)
        complete[:, np.unique(y_true)] = binary
        binary = complete
    ece, _ = top_label_ece(y_true, probabilities)
    return {
        "overall_agreement": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "macro_f1": f1_score(y_true, prediction, average="macro"),
        "macro_auroc_ovr": roc_auc_score(binary, probabilities, average="macro", multi_class="ovr"),
        "macro_auprc": average_precision_score(binary, probabilities, average="macro"),
        "log_loss": log_loss(y_true, probabilities, labels=classes),
        "multiclass_brier": float(np.mean(np.sum((probabilities - binary) ** 2, axis=1))),
        "top_label_ece_10bin": ece,
    }


def block_bootstrap_proxy_metrics(
    y_true: np.ndarray,
    prediction: np.ndarray,
    probabilities: np.ndarray,
    groups: np.ndarray,
    draws: int = 2_000,
    probabilistic_draws: int = 500,
    seed: int = SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    group_indices = {group: np.where(groups == group)[0] for group in unique_groups}
    rows = []
    for draw in range(draws):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        sample = np.concatenate([group_indices[group] for group in sampled_groups])
        y_draw = y_true[sample]
        prediction_draw = prediction[sample]
        probability_draw = probabilities[sample]
        result = {
            "balanced_accuracy": balanced_accuracy_score(y_draw, prediction_draw),
            # Keep the full six-class target when a resampled set omits a class;
            # otherwise the bootstrap denominator changes across draws.
            "macro_f1": f1_score(
                y_draw, prediction_draw, labels=np.arange(probabilities.shape[1]),
                average="macro", zero_division=0,
            ),
            "log_loss": log_loss(y_draw, probability_draw, labels=np.arange(probabilities.shape[1])),
        }
        binary = label_binarize(y_draw, classes=np.arange(probabilities.shape[1]))
        result["multiclass_brier"] = float(np.mean(np.sum((probability_draw - binary) ** 2, axis=1)))
        if draw < probabilistic_draws and len(np.unique(y_draw)) == probabilities.shape[1]:
            result["macro_auroc_ovr"] = roc_auc_score(binary, probability_draw, average="macro", multi_class="ovr")
            result["macro_auprc"] = average_precision_score(binary, probability_draw, average="macro")
        rows.append(result)
    return pd.DataFrame(rows)


def proxy_model_experiment(
    feature_frame: pd.DataFrame,
    output_models: str | Path | None = None,
    profile: str = "publication",
    bootstrap_draws: int = 2_000,
    seed: int = SEED,
    n_jobs: int = -1,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Spatially buffered proxy benchmark selected strictly by inner grouped CV."""
    features = ["blue", "green", "red", "nir08", "swir16", "swir22", "NDVI", "NDBI", "MNDWI", "BSI"]
    # The frozen table's legacy `partition` field predates the final buffered split.
    # Derive the canonical geography only from spatial_block (see docs/audit/PROXY_SPLIT_CONTRACT.md).
    frame = feature_frame.drop(columns=["partition"], errors="ignore").copy()
    frame["block_row"] = frame["spatial_block"] // 10
    frame["block_col"] = frame["spatial_block"] % 10
    test = frame["block_col"].isin([3, 4])
    buffer = frame["block_col"].isin([2, 5])
    development = ~(test | buffer)
    X_dev = frame.loc[development, features].to_numpy()
    y_dev = frame.loc[development, "proxy_class"].to_numpy(dtype=int) - 1
    groups_dev = frame.loc[development, "spatial_block"].to_numpy()
    X_test = frame.loc[test, features].to_numpy()
    y_test = frame.loc[test, "proxy_class"].to_numpy(dtype=int) - 1
    groups_test = frame.loc[test, "spatial_block"].to_numpy()

    if profile == "publication":
        trees = 400
        candidates = [
            (
                "Regularised multinomial logistic",
                Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1_000, solver="lbfgs"))]),
                {"model__C": [0.03, 0.1, 0.3, 1.0, 3.0]},
            ),
            (
                "Random forest",
                RandomForestClassifier(n_estimators=trees, random_state=seed, n_jobs=n_jobs, class_weight="balanced_subsample"),
                {"max_depth": [16, None], "min_samples_leaf": [1, 5, 12]},
            ),
            (
                "Extra trees",
                ExtraTreesClassifier(n_estimators=trees, random_state=seed, n_jobs=n_jobs, class_weight="balanced"),
                {"max_depth": [20, None], "min_samples_leaf": [1, 3, 8]},
            ),
            (
                "Histogram gradient boosting",
                HistGradientBoostingClassifier(max_iter=260, l2_regularization=0.1, random_state=seed),
                {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [31, 63], "min_samples_leaf": [20, 50]},
            ),
            (
                "MLP sensitivity",
                Pipeline(
                    [
                        ("scale", StandardScaler()),
                        ("model", MLPClassifier(max_iter=400, early_stopping=True, random_state=seed)),
                    ]
                ),
                {"model__hidden_layer_sizes": [(64,), (64, 32)], "model__alpha": [0.0001, 0.001, 0.01]},
            ),
        ]
        probabilistic_draws = min(500, bootstrap_draws)
    else:
        trees = 120
        candidates = [
            (
                "Regularised multinomial logistic",
                Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=600, solver="lbfgs"))]),
                {"model__C": [0.1, 1.0]},
            ),
            (
                "Random forest",
                RandomForestClassifier(n_estimators=trees, random_state=seed, n_jobs=n_jobs, class_weight="balanced_subsample"),
                {"max_depth": [16], "min_samples_leaf": [1, 5]},
            ),
            (
                "Extra trees",
                ExtraTreesClassifier(n_estimators=trees, random_state=seed, n_jobs=n_jobs, class_weight="balanced"),
                {"max_depth": [20], "min_samples_leaf": [1, 3]},
            ),
            (
                "Histogram gradient boosting",
                HistGradientBoostingClassifier(max_iter=120, l2_regularization=0.1, random_state=seed),
                {"learning_rate": [0.1], "max_leaf_nodes": [31], "min_samples_leaf": [50]},
            ),
            (
                "MLP sensitivity",
                Pipeline(
                    [
                        ("scale", StandardScaler()),
                        ("model", MLPClassifier(max_iter=220, early_stopping=True, random_state=seed)),
                    ]
                ),
                {"model__hidden_layer_sizes": [(64, 32)], "model__alpha": [0.01]},
            ),
        ]
        probabilistic_draws = min(100, bootstrap_draws)

    comparison_rows = []
    calibration_parts = []
    predictions = frame.loc[test, ["row", "col", "spatial_block", "proxy_class"]].reset_index(drop=True)
    confusion_output = pd.DataFrame()
    fitted = {}
    cv = GroupKFold(n_splits=4)
    for model_index, (name, estimator, grid) in enumerate(candidates):
        # Search serially to avoid nested-process oversubscription; tree estimators
        # may still use ``n_jobs`` internally on Colab/Kaggle CPUs.
        search = GridSearchCV(estimator, grid, scoring="f1_macro", cv=cv, n_jobs=1, refit=True)
        search.fit(X_dev, y_dev, groups=groups_dev)
        prediction = search.predict(X_test)
        probabilities = search.predict_proba(X_test)
        metrics = proxy_metrics(y_test, prediction, probabilities)
        uncertainty = block_bootstrap_proxy_metrics(
            y_test,
            prediction,
            probabilities,
            groups_test,
            draws=bootstrap_draws,
            probabilistic_draws=probabilistic_draws,
            seed=seed + model_index,
        )
        row = {
            "model": name,
            "inner_group_cv_macro_f1": float(search.best_score_),
            **metrics,
            "best_parameters": json.dumps(search.best_params_, sort_keys=True),
            "development_samples": int(development.sum()),
            "buffer_excluded_samples": int(buffer.sum()),
            "test_samples": int(test.sum()),
            "development_blocks": int(frame.loc[development, "spatial_block"].nunique()),
            "buffer_blocks": int(frame.loc[buffer, "spatial_block"].nunique()),
            "test_blocks": int(frame.loc[test, "spatial_block"].nunique()),
            "development_test_block_overlap": int(
                len(set(frame.loc[development, "spatial_block"]) & set(frame.loc[test, "spatial_block"]))
            ),
            "minimum_block_column_separation": 2,
            "validation_role": "WorldCover-derived proxy land-cover agreement; not independent heritage-condition accuracy",
            "block_bootstrap_draws": bootstrap_draws,
            "auroc_auprc_bootstrap_draws": probabilistic_draws,
        }
        for metric in [
            "macro_f1",
            "balanced_accuracy",
            "log_loss",
            "multiclass_brier",
            "macro_auroc_ovr",
            "macro_auprc",
        ]:
            row[f"{metric}_ci_low_95"] = float(uncertainty[metric].quantile(0.025))
            row[f"{metric}_ci_high_95"] = float(uncertainty[metric].quantile(0.975))
        comparison_rows.append(row)
        _, calibration = top_label_ece(y_test, probabilities)
        calibration.insert(0, "model", name)
        calibration_parts.append(calibration)
        predictions[f"predicted_{name}"] = prediction + 1
        predictions[f"confidence_{name}"] = probabilities.max(axis=1)
        fitted[name] = search.best_estimator_

    comparison = pd.DataFrame(comparison_rows).sort_values("inner_group_cv_macro_f1", ascending=False).reset_index(drop=True)
    comparison["primary_selected_from_inner_cv"] = comparison.index == 0
    primary_name = comparison.iloc[0]["model"]
    primary_prediction = predictions[f"predicted_{primary_name}"].to_numpy(dtype=int)
    confusion_output = pd.DataFrame(
        confusion_matrix(frame.loc[test, "proxy_class"], primary_prediction, labels=np.arange(1, 7)),
        index=[f"reference_{value}" for value in range(1, 7)],
        columns=[f"predicted_{value}" for value in range(1, 7)],
    )
    if output_models is not None:
        output_models = Path(output_models)
        output_models.mkdir(parents=True, exist_ok=True)
        joblib.dump(fitted[primary_name], output_models / "primary_proxy_model.joblib")
        (output_models / "primary_proxy_model_metadata.json").write_text(
            json.dumps(
                {
                    "model": primary_name,
                    "feature_order": features,
                    "selection_rule": "highest inner grouped-CV macro-F1; outer proxy test untouched during selection",
                    "inference_boundary": "Proxy land-cover agreement only; not heritage-condition accuracy.",
                },
                indent=2,
            ),
            encoding="utf-8",
        )
    return comparison, pd.concat(calibration_parts, ignore_index=True), confusion_output, predictions


def _subtitle(ax: plt.Axes, text: str) -> None:
    ax.text(
        0.065,
        0.985,
        text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=7.1,
        color=PALETTE["grey"],
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.76, "pad": 1.1},
        zorder=19,
    )


def _panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        0.012,
        0.985,
        label,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        fontweight="bold",
        color="white",
        bbox={"boxstyle": "round,pad=0.22", "facecolor": PALETTE["ink"], "edgecolor": "none", "alpha": 0.92},
        zorder=20,
    )


def _iter_polygon_rings(geometry: dict[str, object]) -> Iterable[np.ndarray]:
    coordinates = geometry["coordinates"]
    polygons = [coordinates] if geometry["type"] == "Polygon" else coordinates
    for polygon in polygons:
        yield np.asarray(polygon[0], dtype=float)


def plot_study_area_map(
    data_root: str | Path,
    components_wgs84: pd.DataFrame,
    terrain_context: dict[str, np.ndarray | float],
    output_dir: str | Path,
) -> plt.Figure:
    fig = plt.figure(figsize=(13.2, 6.7), constrained_layout=True)
    grid = fig.add_gridspec(1, 3, width_ratios=[0.78, 1.46, 0.68])
    locator = fig.add_subplot(grid[0, 0])
    local = fig.add_subplot(grid[0, 1])
    key_axis = fig.add_subplot(grid[0, 2])
    geojson_path = Path(data_root) / "cartography" / "natural_earth_10m_admin0_pakistan_pov_v5_1_1_region.geojson"
    payload = json.loads(geojson_path.read_text(encoding="utf-8"))
    for feature in payload["features"]:
        is_pakistan = feature["properties"].get("ADM0_A3") == "PAK"
        for ring in _iter_polygon_rings(feature["geometry"]):
            if ring[:, 0].max() < 58 or ring[:, 0].min() > 80 or ring[:, 1].max() < 22 or ring[:, 1].min() > 39:
                continue
            locator.fill(
                ring[:, 0],
                ring[:, 1],
                facecolor="#DCEAD9" if is_pakistan else "#F1F3F5",
                edgecolor=PALETTE["ink"] if is_pakistan else "#9AA6B2",
                linewidth=1.0 if is_pakistan else 0.55,
                zorder=2 if is_pakistan else 1,
            )
    site_lon = float(components_wgs84["longitude"].mean())
    site_lat = float(components_wgs84["latitude"].mean())
    locator.scatter(site_lon, site_lat, s=46, marker="*", color=PALETTE["orange"], edgecolor="white", linewidth=0.7, zorder=5)
    locator.annotate("Taxila", (site_lon, site_lat), xytext=(12, -4), textcoords="offset points", fontweight="bold")
    locator.set(xlim=(60, 78.5), ylim=(23, 38.2), xlabel="Longitude (°E)", ylabel="Latitude (°N)")
    locator.set_title("Regional locator")
    _subtitle(locator, "Pakistan point-of-view boundary; contextual cartography only")
    _panel_label(locator, "a")
    locator.set_aspect("equal", adjustable="box")

    dem = np.asarray(terrain_context["dem"])
    latitudes = np.asarray(terrain_context["latitudes"])
    longitudes = np.asarray(terrain_context["longitudes"])
    local.imshow(
        dem,
        extent=[longitudes.min(), longitudes.max(), latitudes.min(), latitudes.max()],
        origin="upper",
        cmap="gist_earth",
        alpha=0.92,
        aspect="auto",
    )
    local.contour(
        longitudes,
        latitudes,
        np.asarray(terrain_context["drainage_mask"]).astype(float),
        levels=[0.5],
        colors=[PALETTE["blue"]],
        linewidths=0.55,
        alpha=0.55,
    )
    for index, point in components_wgs84.reset_index(drop=True).iterrows():
        local.scatter(point["longitude"], point["latitude"], s=24, color=PALETTE["orange"], edgecolor="white", linewidth=0.5, zorder=5)
        local.annotate(str(index + 1), (point["longitude"], point["latitude"]), xytext=(3, 3), textcoords="offset points", fontsize=6.5, zorder=6)
    local.set(xlabel="Longitude (°E)", ylabel="Latitude (°N)")
    local.set_title("Mapped component inventory and terrain context")
    _subtitle(local, "17 mapped official component records; thin blue lines are derived drainage proxies")
    _panel_label(local, "b")
    local.set_aspect(1 / math.cos(math.radians(site_lat)), adjustable="box")
    names = [f"{index + 1}. {name}" for index, name in enumerate(components_wgs84["component_name"])]
    key_axis.axis("off")
    key_axis.text(0.02, 0.98, "Component key", ha="left", va="top", fontsize=9.2, fontweight="bold", color=PALETTE["ink"])
    key_axis.text(0.02, 0.92, "\n".join(names), ha="left", va="top", fontsize=6.7, color=PALETTE["grey"], linespacing=1.28)
    key_axis.text(
        0.02,
        0.04,
        "Saraikala (139-002) remains\nspatially unresolved and is not mapped.",
        ha="left",
        va="bottom",
        fontsize=7,
        color=PALETTE["pink"],
        fontweight="semibold",
    )
    fig.suptitle("Taxila study area and component inventory", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_01_study_area_inventory")
    return fig


def plot_data_integrity(
    data_root: str | Path,
    index_stack: dict[str, np.ndarray],
    output_dir: str | Path,
) -> plt.Figure:
    scenes = pd.read_csv(Path(data_root) / "13_tables" / "retained_landsat_scene_dates.csv")
    scene_counts = scenes.groupby("epoch_id").size().reindex(EPOCHS)
    support_counts = np.asarray([
        np.all(np.isfinite(index_stack[epoch][..., :3]), axis=2).sum() for epoch in EPOCHS
    ])
    common = np.logical_and.reduce([np.all(np.isfinite(index_stack[epoch][..., :3]), axis=2) for epoch in EPOCHS])
    fig, axes = plt.subplots(1, 3, figsize=(11.8, 3.8), constrained_layout=True)
    axes[0].bar(EPOCHS, scene_counts, color=[PALETTE["blue_light"]] * 4 + [PALETTE["blue"]])
    axes[0].bar_label(axes[0].containers[0], padding=2, fontsize=8)
    axes[0].set(title="Retained Landsat scenes", ylabel="Scene count", ylim=(0, max(scene_counts) * 1.18))
    _subtitle(axes[0], "Post-monsoon, three-year epoch windows")
    _panel_label(axes[0], "a")
    axes[1].bar(EPOCHS, 100 * support_counts / (GRID_WIDTH * GRID_HEIGHT), color=PALETTE["teal"])
    axes[1].set(title="Valid spectral support", ylabel="Share of analysis grid (%)", ylim=(90, 100))
    _subtitle(axes[1], "Finite NDVI, NDBI and MNDWI pixels")
    _panel_label(axes[1], "b")
    pieces = [int(common.sum()), int(common.size - common.sum())]
    axes[2].pie(
        pieces,
        labels=["Common support", "Excluded"],
        autopct="%1.2f%%",
        startangle=90,
        colors=[PALETTE["blue"], PALETTE["muted"]],
        wedgeprops={"edgecolor": "white", "linewidth": 1.2},
        textprops={"fontsize": 8},
    )
    axes[2].set_title("Five-epoch common support")
    _subtitle(axes[2], f"{common.sum():,} of {common.size:,} cells retained")
    _panel_label(axes[2], "c")
    fig.suptitle("Input integrity and observation support", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_02_input_integrity")
    return fig


def plot_pressure_maps(
    pressure_grids: dict[str, np.ndarray],
    components_projected: pd.DataFrame,
    output_dir: str | Path,
) -> plt.Figure:
    fig, axes = plt.subplots(2, 3, figsize=(12, 7.1), constrained_layout=True, sharex=True, sharey=True)
    axes = axes.ravel()
    extent = [GRID_XMIN, GRID_XMAX, GRID_YMIN, GRID_YMAX]
    for index, epoch in enumerate(EPOCHS):
        axis = axes[index]
        image_handle = axis.imshow(pressure_grids[epoch], extent=extent, origin="upper", cmap="magma", vmin=0, vmax=1)
        axis.scatter(
            components_projected["x"], components_projected["y"], s=9, facecolor="white", edgecolor=PALETTE["ink"], linewidth=0.35
        )
        axis.set_title(f"{epoch[1:]} epoch")
        _subtitle(axis, "Relative landscape pressure; within-epoch ECDF")
        _panel_label(axis, chr(ord("a") + index))
        axis.ticklabel_format(style="plain")
    axes[5].axis("off")
    colorbar = fig.colorbar(image_handle, ax=axes[:5], orientation="horizontal", fraction=0.045, pad=0.07)
    colorbar.set_label("Relative landscape-pressure score (0–1)")
    fig.supxlabel("Easting, EPSG:32643 (m)")
    fig.supylabel("Northing, EPSG:32643 (m)")
    fig.suptitle("Five-epoch landscape-pressure surfaces", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_03_five_epoch_pressure_maps")
    return fig


def plot_convergence_map(
    convergence: np.ndarray,
    components_projected: pd.DataFrame,
    output_dir: str | Path,
) -> plt.Figure:
    fig, axis = plt.subplots(figsize=(8.6, 6.7), constrained_layout=True)
    colors = ["#F4F6F8", PALETTE["blue_light"], PALETTE["gold"], PALETTE["pink"]]
    cmap = ListedColormap(colors)
    handle = axis.imshow(
        convergence,
        extent=[GRID_XMIN, GRID_XMAX, GRID_YMIN, GRID_YMAX],
        origin="upper",
        cmap=cmap,
        norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N),
    )
    axis.scatter(components_projected["x"], components_projected["y"], s=24, facecolor="white", edgecolor=PALETTE["ink"], linewidth=0.7)
    axis.set(title="Multi-index adverse-tail convergence", xlabel="Easting, EPSG:32643 (m)", ylabel="Northing, EPSG:32643 (m)")
    _subtitle(axis, "Number of NDVI/NDBI/MNDWI adverse-tail criteria met on common support")
    colorbar = fig.colorbar(handle, ax=axis, ticks=[0, 1, 2, 3], fraction=0.045, pad=0.03)
    colorbar.set_label("Convergent criteria")
    fig.suptitle("Spatial screening evidence, E2004 to E2024", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_04_convergence_screen")
    return fig


def plot_terrain_diagnostics(
    terrain_context: dict[str, np.ndarray | float],
    components_wgs84: pd.DataFrame,
    output_dir: str | Path,
) -> plt.Figure:
    fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.5), constrained_layout=True, sharex=True, sharey=True)
    extent = [
        np.asarray(terrain_context["longitudes"]).min(),
        np.asarray(terrain_context["longitudes"]).max(),
        np.asarray(terrain_context["latitudes"]).min(),
        np.asarray(terrain_context["latitudes"]).max(),
    ]
    layers = [
        ("dem", "Terrain elevation", "Elevation (m)", "terrain", None),
        ("slope", "Slope", "Degrees", "viridis", (0, 35)),
        ("wetness", "Topographic wetness proxy", "Relative log index", "YlGnBu", None),
        ("drainage_distance", "Drainage proximity", "Distance to derived drainage (m)", "cividis", (0, 1000)),
    ]
    for index, (axis, layer) in enumerate(zip(axes.ravel(), layers)):
        key, title, label, cmap, limits = layer
        values = np.asarray(terrain_context[key])
        kwargs = {"vmin": limits[0], "vmax": limits[1]} if limits else {}
        handle = axis.imshow(values, extent=extent, origin="upper", cmap=cmap, aspect="auto", **kwargs)
        axis.scatter(components_wgs84["longitude"], components_wgs84["latitude"], s=10, facecolor="white", edgecolor=PALETTE["ink"], linewidth=0.35)
        axis.set_title(title)
        _panel_label(axis, chr(ord("a") + index))
        colorbar = fig.colorbar(handle, ax=axis, fraction=0.045, pad=0.02)
        colorbar.set_label(label)
    fig.supxlabel("Longitude (°E)")
    fig.supylabel("Latitude (°N)")
    fig.suptitle("Terrain and hydrological susceptibility inputs", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_05_terrain_hydrology")
    return fig


def plot_climate_context(
    annual: pd.DataFrame,
    trends: pd.DataFrame,
    output_dir: str | Path,
) -> plt.Figure:
    fig, axes = plt.subplots(2, 2, figsize=(11.2, 7.1), constrained_layout=True)
    panels = [
        ("precipitation_sum_mm", "Annual precipitation", "mm yr⁻¹", PALETTE["blue"]),
        ("temperature_mean_c", "Annual mean temperature", "°C", PALETTE["orange"]),
        ("heavy_precipitation_days", "Heavy-precipitation days", "days yr⁻¹", PALETTE["teal"]),
        ("soil_moisture_mean_m3m3", "Mean soil moisture", "m³ m⁻³", PALETTE["olive"]),
    ]
    for index, (axis, (metric, title, ylabel, color)) in enumerate(zip(axes.ravel(), panels)):
        years = annual["year"].to_numpy()
        values = annual[metric].to_numpy()
        axis.plot(years, values, marker="o", markersize=2.7, linewidth=1.1, color=color, alpha=0.82)
        row = trends[(trends["period"].eq("annual")) & trends["metric"].eq(metric)]
        if len(row):
            row = row.iloc[0]
            fit = row["theil_sen_intercept"] + row["theil_sen_slope_per_year"] * years
            axis.plot(years, fit, color=PALETTE["ink"], linewidth=1.7)
            _subtitle(
                axis,
                f"Theil–Sen {row['theil_sen_slope_per_year']:+.4g} yr⁻¹; block 95% CI "
                f"[{row['moving_block_slope_ci_low_95']:+.4g}, {row['moving_block_slope_ci_high_95']:+.4g}]",
            )
        axis.set(title=title, ylabel=ylabel, xlabel="Year")
        _panel_label(axis, chr(ord("a") + index))
    fig.suptitle("Climate context, 1991–2025", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_06_climate_context")
    return fig


def plot_fixed_priority(
    factors: pd.DataFrame,
    components_projected: pd.DataFrame,
    pressure_2024: np.ndarray,
    output_dir: str | Path,
) -> plt.Figure:
    merged = components_projected.merge(factors, on=["component_id", "component_name"], how="inner")
    merged["rank"] = rank_descending(merged["local_priority_score"], method="ordinal").astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(12.3, 6.2), constrained_layout=True, gridspec_kw={"width_ratios": [1.22, 0.92]})
    extent = [GRID_XMIN, GRID_XMAX, GRID_YMIN, GRID_YMAX]
    axes[0].imshow(pressure_2024, extent=extent, origin="upper", cmap="Greys", vmin=0, vmax=1, alpha=0.58)
    norm = Normalize(vmin=float(merged["local_priority_score"].min()), vmax=float(merged["local_priority_score"].max()))
    cmap = mpl.colormaps["viridis"]
    for record in merged.itertuples(index=False):
        axes[0].add_patch(
            Circle(
                (record.x, record.y),
                500,
                facecolor=cmap(norm(record.local_priority_score)),
                edgecolor="white",
                linewidth=0.75,
                alpha=0.82,
            )
        )
        if record.rank <= 5:
            annotation_offsets = {
                "Giri complex of monuments": (8, 15),
                "Giri Mosque and tombs": (8, -17),
                "Jaulian stupa and monastery": (8, 10),
                "Mohra Moradu stupa and monastery": (8, -13),
                "Dharmarajika stupa and monastery": (-8, 11),
            }
            offset = annotation_offsets.get(record.component_name, (5, 4))
            axes[0].annotate(
                f"{record.rank}. {record.component_name}",
                (record.x, record.y),
                xytext=offset,
                textcoords="offset points",
                fontsize=6.7,
                fontweight="semibold",
                color=PALETTE["ink"],
                ha="right" if offset[0] < 0 else "left",
                arrowprops={"arrowstyle": "-", "color": PALETTE["ink"], "linewidth": 0.45, "alpha": 0.7},
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 1.1},
            )
    axes[0].set(title="Primary 500 m inspection-priority map", xlabel="Easting, EPSG:32643 (m)", ylabel="Northing, EPSG:32643 (m)")
    _subtitle(axes[0], "Analytic circles are not UNESCO or legal property buffers")
    _panel_label(axes[0], "a")
    scalar = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    colorbar = fig.colorbar(scalar, ax=axes[0], fraction=0.045, pad=0.02)
    colorbar.set_label("Local priority P = 0.5L + 0.5T")

    ordered = merged.sort_values("local_priority_score")
    colors = [cmap(norm(value)) for value in ordered["local_priority_score"]]
    axes[1].barh(ordered["component_name"], ordered["local_priority_score"], color=colors)
    axes[1].axvline(ordered["local_priority_score"].median(), color=PALETTE["ink"], linestyle="--", linewidth=0.9, label="Component median")
    axes[1].set(title="Component ranking", xlabel="Local priority score", ylabel="")
    axes[1].set_xlim(max(0, ordered["local_priority_score"].min() - 0.05), min(1, ordered["local_priority_score"].max() + 0.05))
    axes[1].legend(loc="lower right")
    _subtitle(axes[1], "Relative ordering for field inspection—not confirmed deterioration")
    _panel_label(axes[1], "b")
    fig.suptitle("Primary local inspection prioritisation", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_07_fixed_priority")
    return fig


def plot_scale_and_joint_ranks(
    fixed_scores: pd.DataFrame,
    joint_summary: pd.DataFrame,
    output_dir: str | Path,
) -> plt.Figure:
    endpoint = fixed_scores[fixed_scores["epoch_id"].eq("E2024")].copy()
    endpoint["rank"] = endpoint.groupby("radius_m")["local_priority_score"].rank(method="min", ascending=False)
    selected = (
        endpoint[endpoint["radius_m"].eq(500)]
        .nsmallest(8, "rank")["component_id"]
        .tolist()
    )
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.5), constrained_layout=True)
    for component_id in selected:
        local = endpoint[endpoint["component_id"].eq(component_id)].sort_values("radius_m")
        name = local.iloc[0]["component_name"]
        axes[0].plot(local["radius_m"], local["rank"], marker="o", linewidth=1.35, label=name)
    axes[0].invert_yaxis()
    axes[0].set(title="Scale sensitivity of leading components", xlabel="Analytic radius (m)", ylabel="Rank (1 = highest priority)", xticks=RADII)
    axes[0].legend(ncol=2, loc="lower left", frameon=True)
    _subtitle(axes[0], "Ranks recalculated independently at 250, 500 and 1000 m")
    _panel_label(axes[0], "a")

    ordered = joint_summary.sort_values(["rank_median", "probability_top_3"], ascending=[False, True])
    y = np.arange(len(ordered))
    axes[1].hlines(y, ordered["rank_ci_low_95"], ordered["rank_ci_high_95"], color=PALETTE["blue_light"], linewidth=3)
    sizes = 24 + ordered["probability_top_3"].to_numpy() * 75
    scatter = axes[1].scatter(ordered["rank_median"], y, s=sizes, c=ordered["probability_top_3"], cmap="viridis", vmin=0, vmax=1, edgecolor="white", linewidth=0.5)
    axes[1].set_yticks(y, ordered["component_name"])
    axes[1].set(title="Joint spatial + decision uncertainty", xlabel="Rank with 95% interval", ylabel="", xlim=(0.5, len(ordered) + 0.5))
    _subtitle(axes[1], "Point size and colour encode Pr(rank ≤ 3)")
    _panel_label(axes[1], "b")
    colorbar = fig.colorbar(scatter, ax=axes[1], fraction=0.045, pad=0.02)
    colorbar.set_label("Probability of top three")
    fig.suptitle("Scale sensitivity and joint rank uncertainty", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_08_scale_joint_rank_uncertainty")
    return fig


def plot_sensitivity_experiments(
    block_summary: pd.DataFrame,
    jitter_summary: pd.DataFrame,
    weight_summary: pd.DataFrame,
    leading_ids: Iterable[str],
    output_dir: str | Path,
) -> plt.Figure:
    leading_ids = list(leading_ids)
    fig, axes = plt.subplots(1, 3, figsize=(12.4, 4.2), constrained_layout=True, sharey=True)
    configs = [
        (block_summary, "block_size_m", "Shared block size (m)", "Spatial block bootstrap"),
        (jitter_summary, "maximum_displacement_m", "Maximum point displacement (m)", "Coordinate sensitivity"),
        (weight_summary, "alpha", "Symmetric Dirichlet concentration α", "Domain-weight sensitivity"),
    ]
    for panel_index, (axis, (frame, x, xlabel, title)) in enumerate(zip(axes, configs)):
        for component_id in leading_ids:
            local = frame[frame["component_id"].eq(component_id)].sort_values(x)
            if len(local):
                axis.plot(local[x], local["probability_top_3"], marker="o", linewidth=1.4, label=local.iloc[0]["component_name"])
        axis.set(title=title, xlabel=xlabel, ylim=(-0.03, 1.03))
        _panel_label(axis, chr(ord("a") + panel_index))
    axes[0].set_ylabel("Probability of top-three rank")
    axes[2].legend(loc="lower right", fontsize=7)
    fig.suptitle("Sensitivity of top-three inspection priorities", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_09_sensitivity_experiments")
    return fig


def plot_diagnostics(
    factors: pd.DataFrame,
    overlap: pd.DataFrame,
    ablation: pd.DataFrame,
    baselines: pd.DataFrame,
    output_dir: str | Path,
) -> plt.Figure:
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0), constrained_layout=True)
    factor_columns = [
        "ndvi_pressure",
        "ndbi_pressure",
        "mndwi_pressure",
        "slope_pressure",
        "wetness_pressure",
        "drainage_proximity_pressure",
    ]
    labels = ["NDVI", "NDBI", "MNDWI", "Slope", "Wetness", "Drainage"]
    correlation = factors[factor_columns].corr(method="spearman")
    sns.heatmap(correlation, vmin=-1, vmax=1, center=0, cmap="vlag", square=True, annot=True, fmt=".2f", annot_kws={"fontsize": 6.7}, xticklabels=labels, yticklabels=labels, cbar_kws={"label": "Spearman ρ"}, ax=axes[0, 0])
    axes[0, 0].set_title("Factor dependence")
    _panel_label(axes[0, 0], "a")

    local_overlap = overlap[(overlap["radius_m"].eq(500)) & overlap["jaccard_overlap"].gt(0)].nlargest(9, "jaccard_overlap").sort_values("jaccard_overlap")
    pair_names = local_overlap["component_a"].str.replace("Monastery", "Mon.", regex=False) + " – " + local_overlap["component_b"].str.replace("Monastery", "Mon.", regex=False)
    axes[0, 1].barh(pair_names, local_overlap["jaccard_overlap"], color=np.where(local_overlap["jaccard_overlap"].ge(0.25), PALETTE["orange"], PALETTE["blue_light"]))
    axes[0, 1].axvline(0.25, color=PALETTE["ink"], linestyle="--", linewidth=0.9)
    axes[0, 1].set(title="Overlapping 500 m supports", xlabel="Jaccard overlap", ylabel="")
    _subtitle(axes[0, 1], "Dashed threshold flags cluster-aware interpretation")
    _panel_label(axes[0, 1], "b")

    ablation_plot = ablation[~ablation["scenario"].eq("Full hierarchy")].sort_values("mean_absolute_rank_shift")
    axes[1, 0].barh(ablation_plot["scenario"], ablation_plot["mean_absolute_rank_shift"], color=PALETTE["teal"])
    axes[1, 0].set(title="Leave-one-domain/factor-out stability", xlabel="Mean absolute rank shift", ylabel="")
    _panel_label(axes[1, 0], "c")

    baseline_plot = baselines[~baselines["baseline"].eq("Hierarchical reference")].sort_values("spearman_rho_with_reference")
    axes[1, 1].barh(baseline_plot["baseline"], baseline_plot["spearman_rho_with_reference"], color=PALETTE["gold"])
    axes[1, 1].set(title="Alternative scoring baselines", xlabel="Spearman ρ with hierarchical reference", ylabel="", xlim=(-1, 1))
    axes[1, 1].axvline(0, color=PALETTE["ink"], linewidth=0.7)
    _panel_label(axes[1, 1], "d")
    fig.suptitle("Dependence, overlap, ablation and baseline diagnostics", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_10_diagnostics_and_baselines")
    return fig


def plot_proxy_validation(
    comparison: pd.DataFrame,
    calibration: pd.DataFrame,
    confusion: pd.DataFrame,
    output_dir: str | Path,
) -> plt.Figure:
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.25), constrained_layout=True)
    ordered = comparison.sort_values("macro_f1")
    error = np.vstack(
        [ordered["macro_f1"] - ordered["macro_f1_ci_low_95"], ordered["macro_f1_ci_high_95"] - ordered["macro_f1"]]
    )
    axes[0].errorbar(ordered["macro_f1"], ordered["model"], xerr=error, fmt="o", color=PALETTE["blue"], ecolor=PALETTE["blue_light"], capsize=2.5)
    axes[0].set(title="Buffered proxy benchmark (inner-CV selection)", xlabel="Macro-F1 with block-bootstrap 95% CI", ylabel="", xlim=(0.6, 0.9))
    _panel_label(axes[0], "a")

    primary_name = comparison.loc[comparison["primary_selected_from_inner_cv"].astype(bool), "model"].iloc[0]
    local_calibration = calibration[calibration["model"].eq(primary_name)]
    axes[1].plot([0, 1], [0, 1], linestyle="--", color=PALETTE["grey"], linewidth=0.9)
    axes[1].plot(local_calibration["mean_confidence"], local_calibration["accuracy"], marker="o", color=PALETTE["orange"], linewidth=1.4)
    axes[1].set(title="Top-label calibration", xlabel="Mean confidence", ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
    ece = comparison.loc[comparison["model"].eq(primary_name), "top_label_ece_10bin"].iloc[0]
    _subtitle(axes[1], f"{primary_name}; ECE = {ece:.3f}")
    _panel_label(axes[1], "b")

    matrix = confusion.to_numpy(dtype=float)
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    sns.heatmap(normalised, vmin=0, vmax=1, cmap="Blues", annot=True, fmt=".2f", cbar_kws={"label": "Row-normalised share"}, ax=axes[2])
    axes[2].set(title="Primary proxy confusion matrix", xlabel="Predicted proxy class", ylabel="Reference proxy class")
    axes[2].set_xticklabels(range(1, 7))
    axes[2].set_yticklabels(range(1, 7), rotation=0)
    _panel_label(axes[2], "c")
    fig.suptitle("Proxy validation: agreement, calibration and class confusions", fontsize=13, fontweight="bold", color=PALETTE["ink"])
    save_figure(fig, output_dir, "figure_11_proxy_validation")
    return fig


def headline_validation(
    data_root: str | Path,
    tables: dict[str, pd.DataFrame],
    recomputed_scores: pd.DataFrame,
    joint_summary: pd.DataFrame | None = None,
    proxy_comparison: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Machine-check headline manuscript claims against the executable results."""
    root = Path(data_root)
    rows: list[dict[str, object]] = []

    def check(claim: str, observed: float, expected: float, tolerance: float, unit: str, source: str) -> None:
        difference = abs(float(observed) - float(expected))
        rows.append(
            {
                "claim": claim,
                "observed": float(observed),
                "expected_lock": float(expected),
                "absolute_difference": difference,
                "tolerance": tolerance,
                "unit": unit,
                "pass": bool(difference <= tolerance),
                "evidence_source": source,
            }
        )

    endpoint_valid = []
    for epoch in ("E2004", "E2024"):
        indices, _ = load_epoch_indices(root, epoch)
        endpoint_valid.append(np.all(np.isfinite(indices[..., :3]), axis=2))
    common = endpoint_valid[0] & endpoint_valid[1]
    check("E2004–E2024 common spectral support", common.sum(), 386_280, 0, "cells", "reloaded GeoTIFFs")
    check("Common-support share", 100 * common.mean(), 95.73708734014077, 1e-10, "%", "reloaded GeoTIFFs")
    convergence = load_convergence_grid(root)
    valid_convergence = convergence[np.isfinite(convergence)]
    check("Convergence of at least two adverse-tail criteria", np.sum(valid_convergence >= 2), 65_391, 0, "cells", "reloaded F32 grid")
    check("Convergence of all three adverse-tail criteria", np.sum(valid_convergence == 3), 5_493, 0, "cells", "reloaded F32 grid")

    fixed = fixed_factor_frame(recomputed_scores, 500)
    giri = fixed[fixed["component_name"].str.contains("Giri complex", case=False)].iloc[0]
    check("Primary 500 m score: Giri complex", giri["local_priority_score"], 0.5911764705882353, 2e-12, "score", "recomputed component table")
    five_epoch_500 = recomputed_scores[recomputed_scores["radius_m"].eq(500)]
    factor_rho = spearmanr(five_epoch_500["ndvi_pressure"], five_epoch_500["ndbi_pressure"]).statistic
    check("NDVI/NDBI pressure dependence", factor_rho, 0.9717, 5e-5, "Spearman rho", "recomputed factor ranks")

    endpoint = recomputed_scores[recomputed_scores["epoch_id"].eq("E2024")]
    ranks = endpoint.pivot(index="component_id", columns="radius_m", values="local_priority_score").rank(ascending=False)
    check("Scale rank agreement: 500 vs 1000 m", spearmanr(ranks[500], ranks[1000]).statistic, 0.9185, 5e-5, "Spearman rho", "recomputed ranks")
    check("Scale rank agreement: 250 vs 1000 m", spearmanr(ranks[250], ranks[1000]).statistic, 0.7108, 5e-5, "Spearman rho", "recomputed ranks")

    trends = tables["climate_trends"]
    for metric, expected_slope, expected_low, expected_high, expected_fdr in [
        ("precipitation_sum_mm", -10.303704, -17.914978, -2.667900, 0.035982),
        ("temperature_mean_c", 0.041753, 0.009929, 0.073133, 0.034983),
    ]:
        record = trends[(trends["period"].eq("annual")) & trends["metric"].eq(metric)].iloc[0]
        check(f"Annual climate slope: {metric}", record["theil_sen_slope_per_year"], expected_slope, 5e-7, "per year", "frozen block-bootstrap table")
        check(f"Climate CI low: {metric}", record["moving_block_slope_ci_low_95"], expected_low, 5e-7, "per year", "frozen block-bootstrap table")
        check(f"Climate CI high: {metric}", record["moving_block_slope_ci_high_95"], expected_high, 5e-7, "per year", "frozen block-bootstrap table")
        check(f"Climate FDR p: {metric}", record["fdr_adjusted_block_p"], expected_fdr, 5e-6, "p", "frozen block-bootstrap table")

    proxy = proxy_comparison if proxy_comparison is not None else tables["model_frozen"]
    primary = proxy.loc[proxy["primary_selected_from_inner_cv"].astype(bool)].iloc[0]
    check("Primary proxy macro-F1", primary["macro_f1"], 0.830795, 5e-6, "score", "spatially buffered proxy benchmark")
    check("Primary proxy overall agreement", primary["overall_agreement"], 0.833095, 5e-6, "score", "spatially buffered proxy benchmark")
    check("Primary proxy top-label ECE", primary["top_label_ece_10bin"], 0.014280, 5e-6, "score", "spatially buffered proxy benchmark")

    if (
        joint_summary is not None
        and len(joint_summary)
        and int(joint_summary["joint_score_vectors"].iloc[0]) >= 5_000
    ):
        expected_joint = {
            "Giri complex": (2, 0.7332),
            "Bhallar": (3, 0.5280),
            "Giri Mosque": (4, 0.4476),
            "Jaulian": (4, 0.3990),
        }
        for fragment, (rank_expected, top3_expected) in expected_joint.items():
            record = joint_summary[joint_summary["component_name"].str.contains(fragment, case=False)].iloc[0]
            # Stochastic reruns are required to agree substantively, not bit-for-bit.
            check(f"Joint median rank: {fragment}", record["rank_median"], rank_expected, 2.0, "rank", "recomputed joint uncertainty")
            check(f"Joint top-three probability: {fragment}", record["probability_top_3"], top3_expected, 0.12, "probability", "recomputed joint uncertainty")
    return pd.DataFrame(rows)


def evidence_gap_register() -> pd.DataFrame:
    """Explicitly preserve the claim boundary and unresolved evidence gaps."""
    return pd.DataFrame(
        [
            {
                "evidence_item": "Independent field condition/deterioration labels",
                "status": "OPEN",
                "allowed_interpretation": "None; field inspection is the downstream action",
                "prohibited_claim": "Confirmed damage, deterioration or causal heritage risk",
            },
            {
                "evidence_item": "Official UNESCO component polygons",
                "status": "OPEN",
                "allowed_interpretation": "Analytic circles around mapped official component points",
                "prohibited_claim": "Legal, cadastral, property or UNESCO buffer delineation",
            },
            {
                "evidence_item": "Saraikala (139-002) coordinate/geometry",
                "status": "UNRESOLVED",
                "allowed_interpretation": "18 records inventoried; 17 included in spatial ranking",
                "prohibited_claim": "Complete spatial coverage of all 18 component records",
            },
            {
                "evidence_item": "Independent station validation of reanalysis climate",
                "status": "OPEN",
                "allowed_interpretation": "Climate context and robust temporal trend evidence",
                "prohibited_claim": "Site-station measurement equivalence",
            },
            {
                "evidence_item": "External second heritage site",
                "status": "OPEN",
                "allowed_interpretation": "Taxila case-study reproducibility",
                "prohibited_claim": "Demonstrated cross-site transfer accuracy",
            },
            {
                "evidence_item": "WorldCover proxy benchmark",
                "status": "AVAILABLE_PROXY_ONLY",
                "allowed_interpretation": "Spatially buffered land-cover proxy agreement",
                "prohibited_claim": "Independent heritage-condition accuracy",
            },
        ]
    )


def write_output_manifest(output_root: str | Path) -> pd.DataFrame:
    root = Path(output_root)
    rows = []
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.name not in {"SHA256SUMS.txt", "output_manifest.csv"}:
            rows.append(
                {
                    "relative_path": path.relative_to(root).as_posix(),
                    "bytes": path.stat().st_size,
                    "sha256": sha256(path),
                }
            )
    manifest = pd.DataFrame(rows)
    manifest.to_csv(root / "output_manifest.csv", index=False)
    lines = [f"{record.sha256}  {record.relative_path}" for record in manifest.itertuples(index=False)]
    (root / "SHA256SUMS.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return manifest

In [ ]:
configure_publication_style()
np.random.seed(SEED)
random.seed(SEED)

# Optional override, e.g. Path('/content/my_data_folder'). Leave as None for auto-detection.
DATA_ROOT_OVERRIDE = None

# Colab one-click safeguard: if no valid archive/folder is present, open the
# upload chooser. A corrupt or HTML placeholder ZIP is not treated as valid.
if "google.colab" in sys.modules and DATA_ROOT_OVERRIDE is None:
    colab_root = Path("/content")
    folder_ready = not _data_root_missing(colab_root / "Taxila_CHIP_Frozen_Evidence_Data")
    uploaded_archives = list(colab_root.glob("Taxila_CHIP_Colab_Runtime_Data*.zip"))
    uploaded_archives += list(colab_root.glob("Taxila_CHIP_Frozen_Evidence_Data*.zip"))
    valid_archive_ready = any(path.is_file() and zipfile.is_zipfile(path) for path in uploaded_archives)
    if not folder_ready and not valid_archive_ready:
        from google.colab import files
        print("Select Taxila_CHIP_Colab_Runtime_Data.zip (do not upload an HTML shortcut/page).")
        files.upload()

DATA_ROOT = locate_data_root(DATA_ROOT_OVERRIDE)
OUTPUT_ROOT = Path(os.environ.get("TAXILA_OUTPUT_ROOT", Path.cwd() / "Taxila_CHIP_Q1_outputs"))
OUTPUTS = prepare_output_tree(OUTPUT_ROOT)
print("Evidence root:", DATA_ROOT)
print("Output root:", OUTPUTS["root"].resolve())

## 2. Context, data inventory and audit

### Study design

The unit of analysis is an official heritage-component record represented by a documented point. Endpoint spectral pressure is derived from NDVI, NDBI and MNDWI; terrain susceptibility combines slope, wetness and drainage proximity. Percentile normalisation makes the score **relative to the Taxila component set**. Climate is analysed separately as context and is not folded into the local score.

### Frozen evidence sources

- Component records and coordinates: frozen project inventory; CRS **EPSG:32643** for analysis.
- Landsat surface-reflectance composites: five three-year post-monsoon epochs, 30 m grid.
- Terrain: SRTM `.hgt` tile and reproducible D8-derived slope/wetness/drainage proxies.
- Climate: Open-Meteo ERA5-Seamless annual/seasonal summaries for 1991–2025, checked against the frozen product-comparison tables.
- Contextual country outline: [Natural Earth Admin 0 – Countries, Pakistan point of view, v5.1.1](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/). It is not a heritage-property boundary.

In [ ]:
tables = load_core_tables(DATA_ROOT)
components_projected = load_components(DATA_ROOT, projected=True)
components_wgs84 = load_components(DATA_ROOT, projected=False)
indices = {}
index_metadata = {}
for epoch in EPOCHS:
    indices[epoch], index_metadata[epoch] = load_epoch_indices(DATA_ROOT, epoch)
pressure_grids = {epoch: load_pressure_grid(DATA_ROOT, epoch) for epoch in EPOCHS}
convergence = load_convergence_grid(DATA_ROOT)
terrain_context = build_terrain_context(DATA_ROOT, components_wgs84)

inventory_summary = pd.DataFrame(
    [
        ("Official component records", 18, "Inventory"),
        ("Mapped component points", len(components_projected), "Spatial analysis"),
        ("Unresolved component geometry", 1, "Saraikala, 139-002"),
        ("Analysis grid", GRID_WIDTH * GRID_HEIGHT, f"{GRID_WIDTH} × {GRID_HEIGHT} cells"),
        ("Grid resolution", GRID_CELL, "metres"),
        ("Retained Landsat scenes", 55, "11 / 10 / 8 / 8 / 18 by epoch"),
        ("Annual climate years", len(tables["climate_annual"]), "1991–2025"),
    ],
    columns=["item", "value", "note"],
)
display(inventory_summary)
display(components_projected[["component_id", "component_name", "x", "y"]])
inventory_summary.to_csv(OUTPUTS["tables"] / "data_inventory_summary.csv", index=False)

In [ ]:
plot_study_area_map(DATA_ROOT, components_wgs84, terrain_context, OUTPUTS["figures"])
plot_data_integrity(DATA_ROOT, indices, OUTPUTS["figures"])

## 3. Climate context and residual moving-block verification
Annual trends are recomputed using residual blocks: the fitted trend is retained for intervals, while a constant baseline generates null slopes. The annual 12-metric family is adjusted separately from the archived annual/seasonal family. Frozen tables remain labelled historical references; a frozen-value match is not a test of recomputed uncertainty.

In [ ]:
annual_trends_recomputed = moving_block_trend_experiment(
    tables["climate_annual"], draws=CONFIG["climate_draws"], seed=SEED
)
annual_trends_recomputed.to_csv(OUTPUTS["tables"] / "annual_climate_trends_recomputed.csv", index=False)
tables["climate_trends"].to_csv(OUTPUTS["tables"] / "full_climate_trends_frozen_reference.csv", index=False)

climate_headlines = annual_trends_recomputed.query(
    "period == 'annual' and metric in ['precipitation_sum_mm', 'temperature_mean_c', "
    "'heavy_precipitation_days', 'soil_moisture_mean_m3m3']"
)[[
    "metric", "theil_sen_slope_per_year", "moving_block_slope_ci_low_95",
    "moving_block_slope_ci_high_95", "fdr_adjusted_block_p"
]]
display(climate_headlines)
plot_climate_context(tables["climate_annual"], annual_trends_recomputed, OUTPUTS["figures"])
climate_checks = pd.DataFrame([
    {"check": "all p-values within [0,1]", "pass": bool(annual_trends_recomputed["moving_block_p_two_sided"].between(0, 1).all())},
    {"check": "all adjusted p-values within [0,1]", "pass": bool(annual_trends_recomputed["fdr_adjusted_block_p"].between(0, 1).all())},
    {"check": "finite ordered residual-block intervals", "pass": bool((annual_trends_recomputed["moving_block_slope_ci_low_95"] <= annual_trends_recomputed["moving_block_slope_ci_high_95"]).all() and np.isfinite(annual_trends_recomputed[["moving_block_slope_ci_low_95", "moving_block_slope_ci_high_95"]].to_numpy()).all())},
])
climate_checks.to_csv(OUTPUTS["validation"] / "recomputed_climate_validity.csv", index=False)
assert climate_checks["pass"].all(), "Recomputed climate statistics failed validity checks"
display(climate_checks)

## 4. Five-epoch spectral pressure and endpoint convergence

Pressure surfaces are within-epoch relative scores, not physical damage maps. Endpoint convergence counts how many adverse-tail criteria agree: lower NDVI, higher NDBI and lower MNDWI. Threshold sensitivity at $q \in \{0.15,0.20,0.25\}$ is included rather than privileging one cut without disclosure.

In [ ]:
common_endpoint = np.all(np.isfinite(indices["E2004"][..., :3]), axis=2) & np.all(
    np.isfinite(indices["E2024"][..., :3]), axis=2
)
convergence_valid = convergence[np.isfinite(convergence)]
convergence_summary = pd.DataFrame(
    {
        "criterion": ["Common endpoint support", "At least two criteria", "All three criteria"],
        "cells": [common_endpoint.sum(), np.sum(convergence_valid >= 2), np.sum(convergence_valid == 3)],
        "share_percent": [
            100 * common_endpoint.mean(),
            100 * np.mean(convergence_valid >= 2),
            100 * np.mean(convergence_valid == 3),
        ],
    }
)
threshold_sensitivity = pd.read_csv(DATA_ROOT / "q1_revision" / "adverse_tail_threshold_sensitivity.csv")
display(convergence_summary)
display(threshold_sensitivity)
convergence_summary.to_csv(OUTPUTS["tables"] / "endpoint_convergence_summary.csv", index=False)
threshold_sensitivity.to_csv(OUTPUTS["tables"] / "adverse_tail_threshold_sensitivity.csv", index=False)
plot_pressure_maps(pressure_grids, components_projected, OUTPUTS["figures"])
plot_convergence_map(convergence, components_projected, OUTPUTS["figures"])

## 5. Terrain, fixed-score reproduction and primary inspection priority

For component $i$ and support radius $r$:

$$L_{ir}=\tfrac{1}{2}\left[\tfrac{1}{2}(p^{-}_{NDVI}+p^{+}_{NDBI})\right]+\tfrac{1}{2}p^{-}_{MNDWI}$$

$$T_{ir}=\tfrac{1}{3}(p^{+}_{slope}+p^{+}_{wetness}+p^{-}_{drainage\ distance}), \qquad P_{ir}=\tfrac{1}{2}L_{ir}+\tfrac{1}{2}T_{ir}$$

The NDVI/NDBI pair is grouped before MNDWI to reduce double-counting of strongly dependent cover information. Scores are recalculated at 250, 500 and 1000 m; 500 m is primary.

In [ ]:
recomputed_scores = recompute_fixed_scores(tables["landsat"], tables["terrain"])
fixed_reproduction = validate_fixed_reproduction(recomputed_scores, tables["scores_frozen"])
fixed_factors = fixed_factor_frame(recomputed_scores, 500)
fixed_factors["rank"] = rank_descending(fixed_factors["local_priority_score"], method="ordinal").astype(int)
fixed_factors = fixed_factors.sort_values("rank")
display(fixed_reproduction)
display(fixed_factors[["rank", "component_name", "landscape_pressure_score", "terrain_susceptibility_score", "local_priority_score"]])
fixed_reproduction.to_csv(OUTPUTS["validation"] / "fixed_score_reproduction.csv", index=False)
recomputed_scores.to_csv(OUTPUTS["tables"] / "component_epoch_multiscale_scores_recomputed.csv", index=False)
fixed_factors.to_csv(OUTPUTS["tables"] / "primary_500m_component_priority.csv", index=False)
plot_terrain_diagnostics(terrain_context, components_wgs84, OUTPUTS["figures"])
plot_fixed_priority(fixed_factors, components_projected, pressure_grids["E2024"], OUTPUTS["figures"])

## 6. Structural, overlap, ablation and baseline comparisons

This section makes the contribution testable against reasonable alternatives: ungrouped equal-factor scores, CRITIC and entropy weighting, robust-$z$, PCA, landscape-only and terrain-only baselines. It also reports factor removal and overlapping analytic supports so components in the same local cluster are not misread as independent evidence.

In [ ]:
scenarios, structural_stability = structural_scenarios(recomputed_scores)
overlap = support_overlap_table(components_projected)
ablation_summary, ablation_ranks = ablation_experiment(fixed_factors.sort_values("component_id"))
baseline_summary, baseline_ranks = baseline_comparison(
    fixed_factors.sort_values("component_id"),
    tables["landsat"].query("radius_m == 500"),
    tables["terrain"].query("radius_m == 500"),
)
for frame, filename in [
    (scenarios, "structural_scenario_component_ranks.csv"),
    (structural_stability, "structural_scenario_stability.csv"),
    (overlap, "analytic_support_overlap.csv"),
    (ablation_summary, "ablation_summary.csv"),
    (ablation_ranks, "ablation_component_ranks.csv"),
    (baseline_summary, "baseline_comparison_summary.csv"),
    (baseline_ranks, "baseline_component_ranks.csv"),
]:
    frame.to_csv(OUTPUTS["tables"] / filename, index=False)

display(structural_stability.head(10))
display(overlap.query("radius_m == 500 and jaccard_overlap > 0").head(10))
display(baseline_summary.sort_values("spearman_rho_with_reference", ascending=False))
plot_diagnostics(fixed_factors, overlap, ablation_summary, baseline_summary, OUTPUTS["figures"])

## 7. Expanded rank uncertainty

Five uncertainty axes are kept distinct before a joint synthesis:

1. shared support-weighted Bayesian blocks at 90/150/300 m;
2. coordinate displacement up to 0/30/60/120 m, with spectral **and terrain** metrics re-extracted;
3. symmetric domain-weight concentration $\alpha \in \{0.5,1,2\}$;
4. scale and structural alternatives;
5. joint spatial states crossed with multiple decision settings.

The publication profile produces 500 independent spatial states × 10 decision settings = **5,000 joint score vectors**. These are score-vector draws—not 5,000 independent maps or independent sites.

In [ ]:
points_projected = components_projected.sort_values("component_id").reset_index(drop=True)
points_geographic = components_wgs84.sort_values("component_id").reset_index(drop=True)
factors_for_uncertainty = fixed_factor_frame(recomputed_scores, 500).sort_values("component_id").reset_index(drop=True)
baseline_landsat_500 = tables["landsat"].query("radius_m == 500").sort_values(["epoch_id", "component_id"]).reset_index(drop=True)

block_summary, block_rank_draws = shared_block_size_experiment(
    indices["E2024"], points_projected, baseline_landsat_500,
    factors_for_uncertainty["terrain_susceptibility_score"].to_numpy(),
    draws=CONFIG["block_draws"], seed=SEED,
)
jitter_summary, jitter_rank_draws = point_displacement_experiment(
    indices["E2024"], points_projected, points_geographic, terrain_context,
    baseline_landsat_500, draws=CONFIG["jitter_draws"], seed=SEED,
)
weight_summary, weight_rank_draws = weight_concentration_experiment(
    factors_for_uncertainty, draws=CONFIG["weight_draws"], seed=SEED,
)
joint_draws, joint_summary = joint_uncertainty_experiment(
    indices["E2024"], points_projected, points_geographic, terrain_context,
    baseline_landsat_500,
    spatial_states=CONFIG["joint_spatial_states"],
    decision_draws_per_state=CONFIG["decision_draws_per_state"],
    seed=SEED,
)
for frame, filename in [
    (block_summary, "shared_block_size_rank_sensitivity.csv"),
    (jitter_summary, "point_displacement_rank_sensitivity.csv"),
    (weight_summary, "domain_weight_rank_sensitivity.csv"),
    (joint_draws, "joint_uncertainty_component_draws.csv"),
    (joint_summary, "joint_uncertainty_rank_summary.csv"),
]:
    frame.to_csv(OUTPUTS["tables"] / filename, index=False)

display(joint_summary[[
    "component_name", "rank_median", "rank_ci_low_95", "rank_ci_high_95",
    "probability_top_3", "probability_top_5"
]].head(10))
leading_ids = joint_summary.head(4)["component_id"].tolist()
plot_scale_and_joint_ranks(recomputed_scores, joint_summary, OUTPUTS["figures"])
plot_sensitivity_experiments(block_summary, jitter_summary, weight_summary, leading_ids, OUTPUTS["figures"])

## 8. Spatially buffered proxy benchmark

The outer proxy test uses block columns 3–4; adjacent columns 2 and 5 form a no-training buffer; all remaining columns are development data. Model and hyperparameter selection use inner four-fold grouped cross-validation on development data only. The publication profile refits all five model families and uses spatial-block bootstrap intervals. The validation profile reuses the frozen full benchmark to avoid presenting a short engineering run as a new scientific estimate.

In [ ]:
if CONFIG["refit_proxy_models"]:
    proxy_comparison, proxy_calibration, proxy_confusion, proxy_predictions = proxy_model_experiment(
        tables["feature_frame"], OUTPUTS["models"], profile="publication",
        bootstrap_draws=CONFIG["proxy_bootstrap_draws"], seed=SEED,
    )
else:
    proxy_comparison = tables["model_frozen"].copy()
    proxy_calibration = tables["model_calibration_frozen"].copy()
    proxy_confusion = tables["model_confusion_frozen"].set_index(tables["model_confusion_frozen"].columns[0])
    proxy_predictions = pd.DataFrame()

proxy_comparison.to_csv(OUTPUTS["tables"] / "proxy_model_comparison.csv", index=False)
proxy_calibration.to_csv(OUTPUTS["tables"] / "proxy_model_calibration.csv", index=False)
proxy_confusion.to_csv(OUTPUTS["tables"] / "primary_proxy_confusion_matrix.csv")
if len(proxy_predictions):
    proxy_predictions.to_csv(OUTPUTS["tables"] / "proxy_outer_test_predictions.csv", index=False)
display(proxy_comparison[[
    "model", "inner_group_cv_macro_f1", "overall_agreement", "balanced_accuracy", "macro_f1",
    "macro_f1_ci_low_95", "macro_f1_ci_high_95", "macro_auroc_ovr", "macro_auprc",
    "top_label_ece_10bin", "primary_selected_from_inner_cv"
]])
plot_proxy_validation(proxy_comparison, proxy_calibration, proxy_confusion, OUTPUTS["figures"])

## 9. Novelty, comparison and meaningful contribution

The novelty is not “another weighted overlay.” It is the **audit-ready coupling** of heritage-component inspection prioritisation with dependence-aware score construction, multi-axis rank uncertainty, explicit spatial-support overlap, contextual (not causal) climate evidence, spatially buffered proxy benchmarking, and a machine-checkable claim boundary.

In [ ]:
contribution_matrix = pd.DataFrame(
    [
        ("Conventional static weighted overlay", "Single fixed ranking", "CHIP reports scale, block, coordinate, weight, structural and joint rank uncertainty", "Decision-makers see which priorities are robust versus conditional"),
        ("Independent-factor averaging", "Correlated indices can be double-counted", "NDVI/NDBI are grouped before MNDWI; dependence and ablation are reported", "The score hierarchy is inspectable and sensitivity-tested"),
        ("Site points treated independently", "Nearby supports can duplicate local evidence", "Circle-overlap/Jaccard diagnostics flag clustered components", "Field teams can coordinate inspection clusters"),
        ("Random train/test split", "Spatial leakage inflates proxy agreement", "Development–buffer–test block geometry plus inner grouped CV", "Proxy performance has a defensible spatial interpretation"),
        ("Climate folded into a risk score", "Encourages causal overreach", "Climate trends remain contextual and separately qualified", "Contribution stays decision-relevant without claiming causation"),
        ("Map as final answer", "No executable evidence chain", "Notebook exports tables, vector figures, validation checks and SHA-256 manifests", "Every headline can be traced and rerun"),
    ],
    columns=["comparison", "usual limitation", "CHIP contribution", "meaningful decision value"],
)
evidence_gaps = evidence_gap_register()
display(contribution_matrix)
display(evidence_gaps)
contribution_matrix.to_csv(OUTPUTS["tables"] / "novelty_comparison_contribution_matrix.csv", index=False)
evidence_gaps.to_csv(OUTPUTS["validation"] / "evidence_gap_register.csv", index=False)

## 10. Evidence lock, output manifest and takeaways

The next cell checks core counts, scores, rank correlations, climate values and proxy metrics against the manuscript lock. Stochastic joint-rank comparisons use substantive tolerances because a rerun is a Monte Carlo estimate; deterministic counts and fixed scores use exact or near-machine-precision tolerances.

In [ ]:
validation = headline_validation(
    DATA_ROOT, tables, recomputed_scores,
    joint_summary=joint_summary,
    proxy_comparison=proxy_comparison,
)
validation.to_csv(OUTPUTS["validation"] / "headline_evidence_lock.csv", index=False)
display(validation)
print(f"Headline checks passed: {validation['pass'].sum()} / {len(validation)}")
if not validation["pass"].all():
    display(validation.loc[~validation["pass"]])
    warnings.warn("One or more evidence-lock checks require review; do not report a clean reproduction yet.")

run_metadata = {
    "profile": PROFILE,
    "seed": SEED,
    "data_root": "Taxila_CHIP_Frozen_Evidence_Data",
    "analysis_crs": "EPSG:32643",
    "grid": {"width": GRID_WIDTH, "height": GRID_HEIGHT, "cell_m": GRID_CELL},
    "configuration": CONFIG,
    "inference_boundary": "Relative field-inspection priority; not confirmed deterioration, damage, hazard, causality or legal boundary.",
}
(OUTPUTS["validation"] / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
manifest = write_output_manifest(OUTPUTS["root"])
display(manifest.tail())
print("Completed. Outputs:", OUTPUTS["root"].resolve())

## Takeaways

1. Use the leading joint-uncertainty components as an **initial inspection set**, not as a declaration of confirmed damage.
2. Report fixed scores together with rank intervals and top-$k$ probabilities; the uncertainty is part of the result.
3. Treat heavily overlapping 500 m supports as coordinated inspection clusters.
4. Keep the climate finding contextual until field-condition and causal evidence exist.
5. Cite proxy agreement only as WorldCover-derived land-cover validation.
6. The next empirical contribution should be field-labelled condition observations, official component polygons, Saraikala geometry resolution and external-site transfer testing.